# 01 Segmentation And Cutout Only

This notebook does exactly one job: cleanly cut the dog out of the `org` image and the `rep` image, then export standardized intermediate assets for notebooks `02` and `03`.

The main outputs fall into three groups:

1. the dog mask from `org`
2. the `org` background with the dog region removed and left as a hole
3. the `rep` foreground dog as RGBA, alpha, and mask

Design boundaries:

- no pose-driven warping here
- no diffusion refinement here
- this notebook is only about segmentation quality and standardized asset export


## True DragGAN Face Pipeline: Step 01

This notebook is the same segmentation/cutout stage as the main pipeline, but it writes to `data/outputs/face_draggan_pipeline/01_cutout`.

The next notebooks use the replacement cutout from this stage, crop the replacement face, invert that face into an AFHQ Dog StyleGAN latent, run actual DragGAN point dragging in latent space, paste the edited face back into the replacement dog, and then continue with the old body/leg constrained pipeline.

In [ ]:
%matplotlib inline


## Input And Output Contract

This notebook now supports up to **5 example pairs** in one run.

By default it looks for numbered files under `data/example/`:

- `example_01_org.*`, `example_01_rep.*`
- `example_02_org.*`, `example_02_rep.*`
- `...`
- `example_05_org.*`, `example_05_rep.*`

If those numbered pairs are not present, it falls back to the legacy single-pair entry:

- `example_org.*`
- `example_rep.*`

You can still override the input with environment variables for a single custom pair:

- `DOG_REPLACEMENT_ORG_PATH`
- `DOG_REPLACEMENT_REP_PATH`

Output directory:

- `data/outputs/01_cutout/<pair_name>/`

Most important exported files per pair:

- `metadata.json`
- `org_mask.png`
- `org_hole_mask.png`
- `org_background_with_hole.png`
- `rep_mask.png`
- `rep_rgba.png`


In [ ]:
import importlib.util
import json
import math
import os
import subprocess
import sys
from pathlib import Path

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import HTML, clear_output, display
from PIL import Image
from ultralytics import SAM, YOLO


def has_module(module_name):
    return importlib.util.find_spec(module_name) is not None


def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "example").exists() and (candidate / "notebooks").exists():
            return candidate
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "environment.yml").exists() or (candidate / ".git").exists():
            return candidate
    return cwd


def resolve_input_image(label, base_name, env_key, example_dir):
    if os.environ.get(env_key):
        path = Path(os.environ[env_key]).expanduser().resolve()
        if path.exists():
            return path
        raise FileNotFoundError(f"{env_key} is set but does not exist: {path}")
    for suffix in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
        candidate = example_dir / f"{base_name}{suffix}"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"{label} not found. Put it under {example_dir} using base name {base_name}, or set {env_key}."
    )


def resolve_weight(project_root, filename):
    candidates = [
        project_root / "models" / filename,
        project_root / "notebooks" / filename,
        project_root / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return Path(filename)


def resolve_pose_model_path(project_root):
    candidates = [
        Path(os.environ["DOG_POSE_MODEL_PATH"]) if os.environ.get("DOG_POSE_MODEL_PATH") else None,
        project_root / "models" / "dog_pose_best.pt",
        project_root / "dog_pose_best.pt",
        project_root / "data" / "outputs" / "dog_pose_training" / "dog_pose_bootstrap" / "weights" / "best.pt",
    ]
    for candidate in candidates:
        if candidate is not None and candidate.exists():
            return candidate
    return project_root / "models" / "dog_pose_best.pt"


def resolve_example_image(example_dir, stem):
    for suffix in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
        candidate = example_dir / f"{stem}{suffix}"
        if candidate.exists():
            return candidate
    return None


def discover_example_pairs(example_dir, max_pairs=5):
    if os.environ.get("DOG_REPLACEMENT_ORG_PATH") and os.environ.get("DOG_REPLACEMENT_REP_PATH"):
        org_path = Path(os.environ["DOG_REPLACEMENT_ORG_PATH"]).expanduser().resolve()
        rep_path = Path(os.environ["DOG_REPLACEMENT_REP_PATH"]).expanduser().resolve()
        if not org_path.exists() or not rep_path.exists():
            raise FileNotFoundError("Custom environment-path pair does not exist.")
        return [{"pair_id": "custom", "org_path": org_path, "rep_path": rep_path}]

    pair_specs = []
    for idx in range(1, max_pairs + 1):
        org_path = resolve_example_image(example_dir, f"example_{idx:02d}_org")
        rep_path = resolve_example_image(example_dir, f"example_{idx:02d}_rep")
        if org_path is None and rep_path is None:
            continue
        if org_path is None or rep_path is None:
            continue
        pair_specs.append({"pair_id": f"{idx:02d}", "org_path": org_path, "rep_path": rep_path})

    if pair_specs:
        return pair_specs

    org_path = resolve_input_image("Original image", "example_org", "DOG_REPLACEMENT_ORG_PATH", example_dir)
    rep_path = resolve_input_image("Replacement image", "example_rep", "DOG_REPLACEMENT_REP_PATH", example_dir)
    return [{"pair_id": "legacy", "org_path": org_path, "rep_path": rep_path}]


def load_json_if_exists(path):
    path = Path(path)
    if not path.exists():
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def build_pair_name(org_path, rep_path):
    return f"{org_path.stem}__{rep_path.stem}"


def print_pair_header(title):
    print("\n" + "=" * 96)
    print(title)
    print("=" * 96)


if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires a local CUDA GPU, but no CUDA device is available.")

PROJECT_ROOT = find_project_root()
EXAMPLE_DIR = PROJECT_ROOT / "data" / "example"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "outputs" / "face_draggan_pipeline" / "01_cutout"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = 0
FACE_CROP_SCALE = float(os.environ.get("DOG_TRUE_DRAGGAN_FACE_CROP_SCALE", "1.30"))

YOLO_DET_WEIGHTS = resolve_weight(PROJECT_ROOT, "yolov8n.pt")
YOLO_SEG_WEIGHTS = resolve_weight(PROJECT_ROOT, "yolov8x-seg.pt")
SAM_WEIGHTS = resolve_weight(PROJECT_ROOT, "sam_b.pt")
POSE_MODEL_PATH = resolve_pose_model_path(PROJECT_ROOT)

MAM_REPO_DIR = PROJECT_ROOT / "external" / "Matting-Anything"
MAM_CONFIG_PATH = MAM_REPO_DIR / "config" / "MAM-ViTB-8gpu.toml"
MAM_CHECKPOINT_PATH = PROJECT_ROOT / "models" / "matting_anything" / "mam_sam_vitb.pth"
MAM_SAM_CHECKPOINT_PATH = MAM_REPO_DIR / "segment-anything" / "checkpoints" / "sam_vit_b_01ec64.pth"

ENABLE_MATTING_ANYTHING = os.environ.get("DOG_USE_MATTING_ANYTHING", "1") != "0"
MAM_AUTO_SETUP = os.environ.get("DOG_MAM_AUTO_SETUP", "1") != "0"
MAM_HF_REPO_ID = "camenduru/Matting-Anything"
MAM_HF_FILENAME = "mam_sam_vitb.pth"
MAM_SAM_CHECKPOINT_URL = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"

# Matting Anything is only trusted near the replacement-mask edge.
# The inner core remains forced foreground so the matting model cannot delete the dog body.
MAM_CORE_ERODE_KERNEL = int(os.environ.get("DOG_MAM_CORE_ERODE", "15"))
MAM_ALPHA_GAMMA = float(os.environ.get("DOG_MAM_ALPHA_GAMMA", "1.15"))
MAM_SUPPORT_ALPHA_THRESHOLD = float(os.environ.get("DOG_MAM_SUPPORT_ALPHA_THRESHOLD", "0.06"))
MAM_MIN_KEEP_RATIO = float(os.environ.get("DOG_MAM_MIN_KEEP_RATIO", "0.42"))


det_model = YOLO(str(YOLO_DET_WEIGHTS))
seg_model = YOLO(str(YOLO_SEG_WEIGHTS))
sam_model = SAM(str(SAM_WEIGHTS))
pose_model = YOLO(str(POSE_MODEL_PATH))

EXAMPLE_PAIRS_MANIFEST_PATH = EXAMPLE_DIR / "example_pairs_manifest.json"
APPROVAL_MANIFEST_PATH = PROJECT_ROOT / "data" / "outputs" / "00_5_pair_feasibility" / "approved_pairs_manifest.json"
ACTIVE_SESSION_MANIFEST_PATH = OUTPUT_ROOT / "_active_session.json"
ACTIVE_BATCH_MANIFEST_PATH = OUTPUT_ROOT / "_active_batch_manifest.json"
UPLOADED_INPUT_ROOT = OUTPUT_ROOT / "_uploaded_inputs"
EXAMPLE_PAIRS_MANIFEST = load_json_if_exists(EXAMPLE_PAIRS_MANIFEST_PATH) or {}
APPROVAL_MANIFEST = load_json_if_exists(APPROVAL_MANIFEST_PATH) or {}
APPROVALS_BY_PAIR_ID = {item["pair_id"]: item for item in APPROVAL_MANIFEST.get("pairs", [])}
EXAMPLE_ENTRIES_BY_PAIR_ID = {item["pair_id"]: item for item in EXAMPLE_PAIRS_MANIFEST.get("pairs", [])}


def extract_upload_entry(upload_value):
    if isinstance(upload_value, dict):
        values = list(upload_value.values())
    else:
        values = list(upload_value)
    if not values:
        return None
    return values[0]


def get_upload_name(entry):
    if entry is None:
        return None
    return entry.get("name") or entry.get("metadata", {}).get("name") or "uploaded_image.png"


def get_upload_bytes(entry):
    if entry is None:
        return None
    content = entry.get("content")
    if isinstance(content, memoryview):
        return content.tobytes()
    if isinstance(content, bytes):
        return content
    if content is None and "value" in entry:
        value = entry["value"]
        if isinstance(value, memoryview):
            return value.tobytes()
        if isinstance(value, bytes):
            return value
    return bytes(content)


def save_upload_entry(entry, dst_dir: Path, prefix: str):
    dst_dir.mkdir(parents=True, exist_ok=True)
    name = get_upload_name(entry)
    raw = get_upload_bytes(entry)
    suffix = Path(name).suffix or ".png"
    safe_name = "".join(ch if ch.isalnum() or ch in ("-", "_", ".") else "_" for ch in Path(name).stem)
    target = dst_dir / f"{prefix}_{safe_name}{suffix}"
    target.write_bytes(raw)
    return target


def save_active_session_manifest(org_path: Path, rep_path: Path, source_mode="interactive_upload"):
    manifest = {
        "source_mode": source_mode,
        "pair_id": "active_upload",
        "pair_name": build_pair_name(org_path, rep_path),
        "original_path": str(org_path),
        "replacement_path": str(rep_path),
    }
    with open(ACTIVE_SESSION_MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    return manifest


def resolve_pair_specs_for_notebook01():
    active_session = load_json_if_exists(ACTIVE_SESSION_MANIFEST_PATH)
    if active_session:
        org_path = Path(active_session["original_path"]).expanduser().resolve()
        rep_path = Path(active_session["replacement_path"]).expanduser().resolve()
        if not org_path.exists() or not rep_path.exists():
            raise FileNotFoundError(
                "The active uploaded pair manifest exists, but one of the uploaded files is missing. "
                "Upload the pair again or remove the active session manifest."
            )
        return [
            {
                "pair_id": active_session.get("pair_id", "active_upload"),
                "org_path": org_path,
                "rep_path": rep_path,
                "example_manifest_entry": {},
                "approval": None,
                "source_mode": active_session.get("source_mode", "interactive_upload"),
            }
        ], []

    raw_pair_specs = discover_example_pairs(EXAMPLE_DIR, max_pairs=5)
    pair_specs = []
    skipped_pair_specs = []
    for spec in raw_pair_specs:
        pair_entry = EXAMPLE_ENTRIES_BY_PAIR_ID.get(spec["pair_id"], {})
        approval = APPROVALS_BY_PAIR_ID.get(spec["pair_id"])
        spec = {**spec, "example_manifest_entry": pair_entry, "source_mode": "example_pairs"}
        if approval is not None:
            spec["approval"] = approval
            if approval.get("approved_for_01"):
                pair_specs.append(spec)
            else:
                skipped_pair_specs.append(spec)
        else:
            pair_specs.append(spec)
    return pair_specs, skipped_pair_specs


print("Project root:", PROJECT_ROOT)
print("Example dir:", EXAMPLE_DIR)
print("Output root:", OUTPUT_ROOT)
print("CUDA device:", torch.cuda.get_device_name(0))
print("YOLO detector:", YOLO_DET_WEIGHTS)
print("YOLO segmenter:", YOLO_SEG_WEIGHTS)
print("SAM weights:", SAM_WEIGHTS)
print("Pose model:", POSE_MODEL_PATH)
print("Matting Anything enabled:", ENABLE_MATTING_ANYTHING)
print("Matting Anything repo:", MAM_REPO_DIR)
print("Matting Anything checkpoint:", MAM_CHECKPOINT_PATH)
print("Active upload manifest:", ACTIVE_SESSION_MANIFEST_PATH)
print("Notebook 01 will prefer an uploaded pair when the active manifest exists; otherwise it will fall back to the example pairs.")


## Helper Functions

To keep the notebook easy to present, the core utilities are written out in a transparent way:

- image I/O and visualization
- detection-box helpers
- enlarged-context cropping
- mask cleanup with morphology
- `YOLO-seg`, `SAM`, and `GrabCut`

The segmentation strategy is intentionally explicit:

1. find the dog bounding box with `YOLO`
2. run `YOLO-seg` on a larger context crop
3. run `SAM` with a bounding-box prompt
4. use the detected dog bounding box as the replacement foreground prior
5. refine the final mask with `GrabCut`


In [ ]:
def read_rgb(path):
    return np.array(Image.open(path).convert("RGB"))


def save_rgb(path, image):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(np.clip(image, 0, 255).astype(np.uint8)).save(path)


def save_mask(path, mask):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((mask.astype(np.uint8) * 255)).save(path)


def rel_path(path):
    return str(Path(path).resolve().relative_to(PROJECT_ROOT)).replace("\\", "/")


def ensure_uint8(image):
    return np.clip(image, 0, 255).astype(np.uint8)


def plot_images(items, cols=3, figsize=(16, 10)):
    if not items:
        return
    rows = int(math.ceil(len(items) / cols))
    plt.figure(figsize=figsize)
    for idx, (title, image) in enumerate(items, start=1):
        plt.subplot(rows, cols, idx)
        if image.ndim == 2:
            plt.imshow(image, cmap="gray")
        else:
            plt.imshow(image)
        plt.title(title)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


def rgba_on_checkerboard(rgba):
    h, w = rgba.shape[:2]
    tile = 16
    yy, xx = np.indices((h, w))
    board = ((yy // tile + xx // tile) % 2).astype(np.uint8)
    checker = np.where(board[..., None] == 0, 220, 180).astype(np.uint8)
    checker = np.repeat(checker[..., :1], 3, axis=2)
    alpha = rgba[..., 3:4].astype(np.float32) / 255.0
    return ensure_uint8(rgba[..., :3].astype(np.float32) * alpha + checker.astype(np.float32) * (1.0 - alpha))


def show_rgba(rgba, title="RGBA foreground"):
    plt.figure(figsize=(6, 6))
    plt.imshow(rgba_on_checkerboard(rgba))
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


def clip_bbox(bbox, width, height):
    x1, y1, x2, y2 = bbox
    x1 = int(max(0, min(width - 1, math.floor(x1))))
    y1 = int(max(0, min(height - 1, math.floor(y1))))
    x2 = int(max(x1 + 1, min(width, math.ceil(x2))))
    y2 = int(max(y1 + 1, min(height, math.ceil(y2))))
    return [x1, y1, x2, y2]


def bbox_area(bbox):
    x1, y1, x2, y2 = bbox
    return max(1, x2 - x1) * max(1, y2 - y1)


def bbox_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    return inter / max(1, bbox_area(box_a) + bbox_area(box_b) - inter)


def mask_to_bbox(mask):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    return [int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1]


def draw_bbox(image, bbox, color=(255, 80, 80), thickness=4):
    out = image.copy()
    x1, y1, x2, y2 = [int(v) for v in bbox]
    cv2.rectangle(out, (x1, y1), (x2, y2), color, thickness)
    return out


def overlay_mask(image, mask, color=(30, 144, 255), alpha=0.45):
    out = image.astype(np.float32).copy()
    color_arr = np.array(color, dtype=np.float32)
    out[mask] = out[mask] * (1.0 - alpha) + color_arr * alpha
    return ensure_uint8(out)


def expand_bbox(bbox, width, height, margin_ratio=0.18, min_margin=16):
    x1, y1, x2, y2 = bbox
    bw = x2 - x1
    bh = y2 - y1
    margin = int(max(min_margin, round(max(bw, bh) * margin_ratio)))
    return clip_bbox([x1 - margin, y1 - margin, x2 + margin, y2 + margin], width, height)


def crop_with_context(image, bbox, margin_ratio=0.18):
    h, w = image.shape[:2]
    crop_bbox = expand_bbox(bbox, w, h, margin_ratio=margin_ratio)
    x1, y1, x2, y2 = crop_bbox
    crop = image[y1:y2, x1:x2].copy()
    local_bbox = [bbox[0] - x1, bbox[1] - y1, bbox[2] - x1, bbox[3] - y1]
    return crop, crop_bbox, local_bbox


def uncrop_mask(mask_crop, crop_bbox, full_shape):
    full_mask = np.zeros(full_shape[:2], dtype=bool)
    x1, y1, x2, y2 = crop_bbox
    full_mask[y1:y2, x1:x2] = mask_crop.astype(bool)
    return full_mask


def refine_binary_mask(mask, close_kernel=9, open_kernel=3):
    mask_u8 = mask.astype(np.uint8)
    if close_kernel > 0:
        kernel = np.ones((close_kernel, close_kernel), np.uint8)
        mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_CLOSE, kernel)
    if open_kernel > 0:
        kernel = np.ones((open_kernel, open_kernel), np.uint8)
        mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_OPEN, kernel)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    if num_labels <= 1:
        return mask_u8.astype(bool)
    keep = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return (labels == keep)


def estimate_foreground_mask_from_borders(image, pad_ratio=0.04):
    h, w = image.shape[:2]
    pad_y = max(4, int(round(h * pad_ratio)))
    pad_x = max(4, int(round(w * pad_ratio)))
    border_pixels = np.concatenate(
        [
            image[:pad_y].reshape(-1, 3),
            image[-pad_y:].reshape(-1, 3),
            image[:, :pad_x].reshape(-1, 3),
            image[:, -pad_x:].reshape(-1, 3),
        ],
        axis=0,
    ).astype(np.float32)
    bg_color = np.median(border_pixels, axis=0)
    diff = np.linalg.norm(image.astype(np.float32) - bg_color[None, None, :], axis=2)
    threshold = max(18.0, float(np.percentile(diff, 72)))
    mask = diff > threshold
    return refine_binary_mask(mask, close_kernel=9, open_kernel=3)



def dog_class_id(model):
    dog_ids = [idx for idx, name in model.names.items() if name == "dog"]
    if not dog_ids:
        raise RuntimeError("The YOLO model does not expose a dog class.")
    return dog_ids[0]


def detect_best_dog(image, conf=0.18):
    results = det_model.predict(image, conf=conf, verbose=False, device=DEVICE)
    result = results[0]
    if result.boxes is None or len(result.boxes) == 0:
        raise RuntimeError("YOLO detector did not return any object boxes.")
    target_cls = dog_class_id(det_model)
    h, w = image.shape[:2]
    dog_candidates = []
    fallback_candidates = []
    for box in result.boxes:
        xyxy = clip_bbox(box.xyxy[0].detach().cpu().numpy().tolist(), w, h)
        score = float(box.conf.item())
        area = bbox_area(xyxy)
        cls_id = int(box.cls.item())
        item = (xyxy, score, area, cls_id)
        fallback_candidates.append(item)
        if cls_id == target_cls:
            dog_candidates.append(item)
    candidates = dog_candidates if dog_candidates else fallback_candidates
    bbox, score, _, cls_id = max(candidates, key=lambda item: item[1] * math.sqrt(item[2]))
    if dog_candidates:
        return bbox, score
    predicted_name = det_model.names.get(cls_id, str(cls_id))
    print(f"YOLO did not label any box as dog; falling back to the strongest object box classified as {predicted_name}.")
    return bbox, score


def segment_with_yolo_seg(image, bbox, conf=0.12, context_ratio=0.22):
    crop, crop_bbox, local_bbox = crop_with_context(image, bbox, margin_ratio=context_ratio)
    results = seg_model.predict(crop, conf=conf, verbose=False, device=DEVICE)
    result = results[0]
    if result.masks is None or result.boxes is None or len(result.boxes) == 0:
        return None, crop_bbox
    target_cls = dog_class_id(seg_model)
    masks = result.masks.data.detach().cpu().numpy()
    h, w = crop.shape[:2]
    dog_candidates = []
    fallback_candidates = []
    for i, box in enumerate(result.boxes):
        mask = masks[i]
        if mask.shape != (h, w):
            mask = cv2.resize(mask.astype(np.float32), (w, h), interpolation=cv2.INTER_LINEAR)
        mask_bool = mask > 0.45
        if mask_bool.sum() < 20:
            continue
        local_box = clip_bbox(box.xyxy[0].detach().cpu().numpy().tolist(), w, h)
        focus = bbox_iou(local_box, local_bbox)
        full_mask = uncrop_mask(mask_bool, crop_bbox, image.shape)
        cls_id = int(box.cls.item())
        item = (full_mask, float(box.conf.item()), focus, cls_id)
        fallback_candidates.append(item)
        if cls_id == target_cls:
            dog_candidates.append(item)
    candidates = dog_candidates if dog_candidates else fallback_candidates
    if not candidates:
        return None, crop_bbox
    best_mask, _, _, cls_id = max(candidates, key=lambda item: item[0].sum() * (0.3 + item[1]) * (0.2 + item[2]))
    if not dog_candidates:
        predicted_name = seg_model.names.get(cls_id, str(cls_id))
        print(f"YOLO-seg did not label any mask as dog; falling back to the strongest mask classified as {predicted_name}.")
    return refine_binary_mask(best_mask, close_kernel=9, open_kernel=3), crop_bbox


def score_mask_with_prior(mask, prior_mask):
    inter = np.logical_and(mask, prior_mask).sum()
    outside = np.logical_and(mask, ~prior_mask).sum()
    prior_area = max(1, prior_mask.sum())
    mask_area = max(1, mask.sum())
    return inter / prior_area + inter / mask_area - 0.75 * outside / mask_area


def segment_with_sam(image, bbox, prior_mask=None, context_ratio=0.22, max_side=1280):
    crop, crop_bbox, local_bbox = crop_with_context(image, bbox, margin_ratio=context_ratio)
    h, w = crop.shape[:2]
    scale = min(1.0, max_side / max(h, w))
    if scale < 1.0:
        infer_w = int(round(w * scale))
        infer_h = int(round(h * scale))
        infer_image = cv2.resize(crop, (infer_w, infer_h), interpolation=cv2.INTER_AREA)
        infer_bbox = [int(round(v * scale)) for v in local_bbox]
    else:
        infer_image = crop
        infer_bbox = local_bbox
    results = sam_model.predict(infer_image, bboxes=[infer_bbox], verbose=False, device=DEVICE)
    result = results[0]
    if result.masks is None or len(result.masks.data) == 0:
        return None, crop_bbox
    masks = result.masks.data.detach().cpu().numpy()
    candidates = []
    for mask in masks:
        if mask.shape != infer_image.shape[:2]:
            mask = cv2.resize(mask.astype(np.float32), (infer_image.shape[1], infer_image.shape[0]), interpolation=cv2.INTER_NEAREST)
        if scale < 1.0:
            mask = cv2.resize(mask.astype(np.float32), (w, h), interpolation=cv2.INTER_NEAREST)
        full_mask = uncrop_mask(mask > 0.5, crop_bbox, image.shape)
        candidates.append(full_mask)
    if not candidates:
        return None, crop_bbox
    if prior_mask is not None:
        best = max(candidates, key=lambda item: score_mask_with_prior(item, prior_mask))
        prior_soft = cv2.dilate(prior_mask.astype(np.uint8), np.ones((17, 17), np.uint8), iterations=1).astype(bool)
        constrained = np.logical_and(best, prior_soft)
        if constrained.sum() > 0.35 * max(1, prior_mask.sum()):
            best = constrained
    else:
        x1, y1, x2, y2 = bbox
        best = max(candidates, key=lambda item: int(item[y1:y2, x1:x2].sum()))
    return refine_binary_mask(best, close_kernel=7, open_kernel=3), crop_bbox


def refine_mask_grabcut(image, mask, iter_count=5):
    if mask is None or mask.sum() < 10:
        return mask
    mask_u8 = mask.astype(np.uint8)
    eroded = cv2.erode(mask_u8, np.ones((5, 5), np.uint8), iterations=1)
    dilated = cv2.dilate(mask_u8, np.ones((21, 21), np.uint8), iterations=1)
    gc_mask = np.full(mask.shape, cv2.GC_BGD, dtype=np.uint8)
    gc_mask[dilated > 0] = cv2.GC_PR_FGD
    gc_mask[eroded > 0] = cv2.GC_FGD
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)
    bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    try:
        cv2.grabCut(bgr, gc_mask, None, bgd_model, fgd_model, iter_count, cv2.GC_INIT_WITH_MASK)
        refined = np.logical_or(gc_mask == cv2.GC_FGD, gc_mask == cv2.GC_PR_FGD)
        if refined.sum() > 0.25 * mask.sum():
            return refine_binary_mask(refined, close_kernel=7, open_kernel=3)
    except Exception as exc:
        print("GrabCut refinement skipped:", type(exc).__name__, exc)
    return refine_binary_mask(mask, close_kernel=7, open_kernel=3)


def refine_replacement_mask_conservative(
    image,
    mask,
    iter_count=3,
    erode_kernel=3,
    dilate_kernel=9,
    support_growth_kernel=11,
    min_keep_ratio=0.72,
):
    if mask is None or mask.sum() < 10:
        return mask
    mask_u8 = mask.astype(np.uint8)
    eroded = cv2.erode(mask_u8, np.ones((int(erode_kernel), int(erode_kernel)), np.uint8), iterations=1)
    dilated = cv2.dilate(mask_u8, np.ones((int(dilate_kernel), int(dilate_kernel)), np.uint8), iterations=1)
    gc_mask = np.full(mask.shape, cv2.GC_BGD, dtype=np.uint8)
    gc_mask[dilated > 0] = cv2.GC_PR_FGD
    gc_mask[eroded > 0] = cv2.GC_FGD
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)
    bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    refined = mask.copy()
    try:
        cv2.grabCut(bgr, gc_mask, None, bgd_model, fgd_model, int(iter_count), cv2.GC_INIT_WITH_MASK)
        refined = np.logical_or(gc_mask == cv2.GC_FGD, gc_mask == cv2.GC_PR_FGD)
    except Exception as exc:
        print("Conservative replacement GrabCut skipped:", type(exc).__name__, exc)

    growth_limit = cv2.dilate(mask_u8, np.ones((int(support_growth_kernel), int(support_growth_kernel)), np.uint8), iterations=1) > 0
    constrained = np.logical_and(refined, growth_limit)
    constrained = refine_binary_mask(constrained, close_kernel=5, open_kernel=0)

    min_keep_pixels = max(25, int(round(float(mask.sum()) * float(min_keep_ratio))))
    if constrained.sum() < min_keep_pixels:
        constrained = refine_binary_mask(mask, close_kernel=5, open_kernel=0)

    return constrained


REP_MASK_INSET_KERNEL = 5
REP_MASK_INSET_ITERATIONS = 1
REP_MASK_INSET_MIN_AREA_RATIO = 0.72
REP_CORE_MASK_KERNEL = 13
REP_ALPHA_FEATHER_RADIUS = 6.0
REP_EDGE_BG_RING_KERNEL = 19
REP_EDGE_DECONTAM_STRENGTH = 0.92
REP_EDGE_ALPHA_FLOOR = 0.35

# Replacement-boundary refinement is modeled as a trimap problem:
# a confident foreground core, a confident background exterior,
# and a narrow unknown band where edge refinement is allowed.
REP_TRIMAP_FG_KERNEL = 11
REP_TRIMAP_BG_KERNEL = 21
REP_TRIMAP_ITER_COUNT = 3
REP_TRIMAP_ALPHA_THRESHOLD = 0.62
REP_ALPHA_SUPPORT_FLOOR = 0.02
REP_ALPHA_CORE_THRESHOLD = 0.96
REP_ALPHA_BLUR_KERNEL = 5
REP_EXPORT_ALPHA_CUTOFF = 0.08


def inset_mask_inward(
    mask,
    kernel_size=REP_MASK_INSET_KERNEL,
    iterations=REP_MASK_INSET_ITERATIONS,
    min_area_ratio=REP_MASK_INSET_MIN_AREA_RATIO,
):
    if mask is None or mask.sum() < 10 or kernel_size <= 1 or iterations <= 0:
        return mask
    kernel = np.ones((int(kernel_size), int(kernel_size)), np.uint8)
    inset = cv2.erode(mask.astype(np.uint8), kernel, iterations=int(iterations)) > 0
    min_pixels = max(25, int(round(float(mask.sum()) * float(min_area_ratio))))
    if inset.sum() < min_pixels:
        return mask.copy()
    return refine_binary_mask(inset, close_kernel=3, open_kernel=0)


def build_mask_layers(crop_mask, core_kernel=REP_CORE_MASK_KERNEL, feather_radius=REP_ALPHA_FEATHER_RADIUS):
    crop_mask = crop_mask.astype(bool)
    if crop_mask.sum() < 10:
        alpha = crop_mask.astype(np.float32)
        return crop_mask.copy(), np.zeros_like(crop_mask, dtype=bool), alpha
    core_mask = inset_mask_inward(
        crop_mask,
        kernel_size=core_kernel,
        iterations=1,
        min_area_ratio=0.45,
    )
    if core_mask.sum() < max(25, int(round(crop_mask.sum() * 0.35))):
        core_mask = crop_mask.copy()
    boundary_ring = np.logical_and(crop_mask, ~core_mask)
    dist_in = cv2.distanceTransform(crop_mask.astype(np.uint8), cv2.DIST_L2, 5).astype(np.float32)
    alpha = np.clip(dist_in / max(1.0, float(feather_radius)), 0.0, 1.0)
    alpha[core_mask] = 1.0
    alpha[~crop_mask] = 0.0
    return core_mask, boundary_ring, alpha


def estimate_background_rgb_around_mask(crop_rgb, crop_mask, ring_kernel=REP_EDGE_BG_RING_KERNEL):
    kernel = np.ones((int(ring_kernel), int(ring_kernel)), np.uint8)
    outer_ring = np.logical_and(
        cv2.dilate(crop_mask.astype(np.uint8), kernel, iterations=1).astype(bool),
        ~crop_mask,
    )
    if outer_ring.sum() < 20:
        h, w = crop_mask.shape
        border = np.zeros_like(crop_mask, dtype=bool)
        border[:2] = True
        border[-2:] = True
        border[:, :2] = True
        border[:, -2:] = True
        outer_ring = border
    bg_pixels = crop_rgb[outer_ring]
    if bg_pixels.size == 0:
        bg_pixels = crop_rgb.reshape(-1, 3)
    return np.median(bg_pixels.astype(np.float32), axis=0)


def decontaminate_boundary_rgb(
    crop_rgb,
    crop_mask,
    alpha,
    boundary_ring,
    strength=REP_EDGE_DECONTAM_STRENGTH,
    alpha_floor=REP_EDGE_ALPHA_FLOOR,
):
    if boundary_ring.sum() < 10:
        return crop_rgb.copy(), None
    bg_rgb = estimate_background_rgb_around_mask(crop_rgb, crop_mask)
    rgb = crop_rgb.astype(np.float32)
    alpha3 = alpha[..., None].astype(np.float32)
    safe_alpha = np.clip(alpha3, float(alpha_floor), 1.0)
    recovered = (rgb - (1.0 - alpha3) * bg_rgb.reshape(1, 1, 3)) / safe_alpha
    recovered = np.clip(recovered, 0.0, 255.0)
    ring_weight = boundary_ring[..., None].astype(np.float32) * np.clip((1.0 - alpha3) * 1.35 + 0.15, 0.0, 1.0)
    weight = np.clip(float(strength) * ring_weight, 0.0, 1.0)
    decontaminated = rgb * (1.0 - weight) + recovered * weight
    decontaminated[~crop_mask] = rgb[~crop_mask]
    return np.clip(decontaminated, 0.0, 255.0).astype(np.uint8), bg_rgb


def visualize_trimap(trimap):
    trimap_vis = np.zeros((*trimap.shape, 3), dtype=np.uint8)
    trimap_vis[trimap == 0] = (30, 30, 30)
    trimap_vis[trimap == 128] = (255, 210, 0)
    trimap_vis[trimap == 255] = (255, 255, 255)
    return trimap_vis


def build_trimap_from_mask(mask, fg_kernel=REP_TRIMAP_FG_KERNEL, bg_kernel=REP_TRIMAP_BG_KERNEL):
    mask = mask.astype(np.uint8)
    if mask.sum() < 10:
        trimap = np.zeros(mask.shape, dtype=np.uint8)
        return trimap, mask.astype(bool), np.ones(mask.shape, dtype=bool), np.zeros(mask.shape, dtype=bool)
    fg = cv2.erode(mask, np.ones((int(fg_kernel), int(fg_kernel)), np.uint8), iterations=1) > 0
    bg = cv2.dilate(mask, np.ones((int(bg_kernel), int(bg_kernel)), np.uint8), iterations=1) == 0
    unknown = ~(fg | bg)
    trimap = np.full(mask.shape, 128, dtype=np.uint8)
    trimap[bg] = 0
    trimap[fg] = 255
    return trimap, fg, bg, unknown


def alpha_from_trimap_and_mask(
    refined_mask,
    sure_fg,
    sure_bg,
    blur_kernel=REP_ALPHA_BLUR_KERNEL,
    support_floor=REP_ALPHA_SUPPORT_FLOOR,
):
    refined_mask = refined_mask.astype(bool)
    sure_fg = np.logical_and(sure_fg.astype(bool), refined_mask)
    expanded_support = cv2.dilate(refined_mask.astype(np.uint8), np.ones((5, 5), np.uint8), iterations=1) > 0
    sure_bg = np.logical_or(sure_bg.astype(bool), ~expanded_support)

    dist_to_fg = cv2.distanceTransform((~sure_fg).astype(np.uint8), cv2.DIST_L2, 5).astype(np.float32)
    dist_to_bg = cv2.distanceTransform((~sure_bg).astype(np.uint8), cv2.DIST_L2, 5).astype(np.float32)
    alpha = dist_to_bg / np.maximum(1e-6, dist_to_fg + dist_to_bg)
    alpha[sure_fg] = 1.0
    alpha[sure_bg] = 0.0

    if blur_kernel and int(blur_kernel) > 1:
        k = int(blur_kernel)
        if k % 2 == 0:
            k += 1
        alpha = cv2.GaussianBlur(alpha, (k, k), 0)
        alpha[sure_fg] = 1.0
        alpha[sure_bg] = 0.0

    alpha = np.clip(alpha, 0.0, 1.0)
    alpha[np.logical_and(~refined_mask, alpha < support_floor)] = 0.0
    unknown_band = np.logical_and(alpha > support_floor, alpha < 0.999)
    return alpha, unknown_band


def binary_mask_from_soft_alpha(alpha, threshold=REP_TRIMAP_ALPHA_THRESHOLD):
    mask = alpha >= float(threshold)
    return refine_binary_mask(mask, close_kernel=5, open_kernel=0)


def core_and_boundary_from_alpha(
    alpha,
    support_mask=None,
    support_floor=REP_ALPHA_SUPPORT_FLOOR,
    core_threshold=REP_ALPHA_CORE_THRESHOLD,
):
    support = (alpha > float(support_floor)) if support_mask is None else support_mask.astype(bool)
    core = np.logical_and(alpha >= float(core_threshold), support)
    if core.sum() < max(25, int(round(support.sum() * 0.2))):
        core = inset_mask_inward(support, kernel_size=REP_CORE_MASK_KERNEL, iterations=1, min_area_ratio=0.3)
    boundary = np.logical_and(support, ~core)
    return support, core, boundary


def refine_replacement_mask_with_trimap(image, coarse_mask, iter_count=REP_TRIMAP_ITER_COUNT):
    coarse_mask = refine_binary_mask(coarse_mask, close_kernel=7, open_kernel=0)
    trimap, sure_fg, sure_bg, unknown = build_trimap_from_mask(coarse_mask)

    gc_mask = np.full(coarse_mask.shape, cv2.GC_PR_BGD, dtype=np.uint8)
    gc_mask[trimap == 0] = cv2.GC_BGD
    gc_mask[trimap == 255] = cv2.GC_FGD
    gc_mask[np.logical_and(trimap == 128, coarse_mask)] = cv2.GC_PR_FGD

    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)
    bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    try:
        cv2.grabCut(bgr, gc_mask, None, bgd_model, fgd_model, int(iter_count), cv2.GC_INIT_WITH_MASK)
        refined_mask = np.logical_or(gc_mask == cv2.GC_FGD, gc_mask == cv2.GC_PR_FGD)
    except Exception as exc:
        print("Trimap refinement skipped:", type(exc).__name__, exc)
        refined_mask = coarse_mask.copy()

    if refined_mask.sum() < max(25, int(round(coarse_mask.sum() * 0.6))):
        refined_mask = coarse_mask.copy()
    refined_mask = refine_binary_mask(refined_mask, close_kernel=5, open_kernel=0)

    alpha_soft, unknown_band = alpha_from_trimap_and_mask(refined_mask, sure_fg, sure_bg)
    support_mask = binary_mask_from_soft_alpha(alpha_soft)
    if support_mask.sum() < max(25, int(round(refined_mask.sum() * 0.65))):
        support_mask = refined_mask.copy()
    support_mask, core_mask, boundary_ring = core_and_boundary_from_alpha(alpha_soft, support_mask=support_mask)

    return {
        "trimap": trimap,
        "sure_fg": sure_fg,
        "sure_bg": sure_bg,
        "unknown_seed_band": unknown,
        "refined_mask": refined_mask,
        "support_mask": support_mask,
        "core_mask": core_mask,
        "boundary_ring": boundary_ring,
        "soft_alpha": alpha_soft,
        "soft_unknown_band": unknown_band,
    }


def local_mask_to_full(local_mask, bbox, full_shape):
    full_mask = np.zeros(full_shape[:2], dtype=bool)
    x1, y1, x2, y2 = [int(v) for v in bbox]
    full_mask[y1:y2, x1:x2] = local_mask.astype(bool)
    return full_mask


def extract_rgba(
    image,
    mask,
    return_details=False,
    decontaminate_edges=False,
    alpha_override=None,
    core_mask_override=None,
    boundary_ring_override=None,
):
    support_mask = mask.astype(bool)
    bbox = mask_to_bbox(support_mask)
    if bbox is None:
        raise RuntimeError("Cannot extract RGBA from an empty mask.")
    x1, y1, x2, y2 = bbox
    crop_rgb = image[y1:y2, x1:x2].copy()
    crop_mask = support_mask[y1:y2, x1:x2].copy().astype(bool)

    if alpha_override is not None:
        alpha = np.clip(alpha_override[y1:y2, x1:x2].astype(np.float32), 0.0, 1.0)
        # The soft alpha is estimated on the full image and can remain non-zero
        # in the trimap unknown band even outside the final support mask. The
        # exported RGBA must obey the final support mask, otherwise background
        # pixels leak into the replacement crop.
        alpha[~crop_mask] = 0.0
        alpha[alpha < REP_EXPORT_ALPHA_CUTOFF] = 0.0
        if core_mask_override is not None:
            core_mask = core_mask_override[y1:y2, x1:x2].copy().astype(bool)
        else:
            core_mask = alpha >= REP_ALPHA_CORE_THRESHOLD
        if boundary_ring_override is not None:
            boundary_ring = boundary_ring_override[y1:y2, x1:x2].copy().astype(bool)
        else:
            boundary_ring = np.logical_and(crop_mask, ~core_mask)
    else:
        core_mask, boundary_ring, alpha = build_mask_layers(crop_mask)

    bg_rgb = None
    crop_rgb_out = crop_rgb.copy()
    if decontaminate_edges:
        crop_rgb_out, bg_rgb = decontaminate_boundary_rgb(crop_rgb_out, crop_mask, alpha, boundary_ring)
    # Keep fully transparent RGB neutral/empty. This does not change alpha
    # compositing, but prevents later stages that inspect RGB directly from
    # seeing stale background colors in transparent pixels.
    crop_rgb_out[alpha <= 0.0] = 0
    rgba = np.dstack([crop_rgb_out, np.uint8(np.clip(alpha, 0.0, 1.0) * 255)])
    if return_details:
        return {
            "rgb": crop_rgb_out,
            "mask": crop_mask,
            "rgba": rgba,
            "bbox": bbox,
            "core_mask": core_mask,
            "boundary_ring": boundary_ring,
            "alpha": alpha,
            "edge_background_rgb": None if bg_rgb is None else [float(v) for v in bg_rgb],
        }
    return crop_rgb_out, crop_mask, rgba, bbox


MAM_MODEL_CACHE = None


def ensure_odd_kernel(value, minimum=1):
    value = max(int(value), int(minimum))
    if value % 2 == 0:
        value += 1
    return value


def install_python_package_if_missing(module_name, pip_name=None):
    if has_module(module_name):
        return
    if not MAM_AUTO_SETUP:
        raise ImportError(
            f"Missing optional package {module_name!r}. Install it or set MAM_AUTO_SETUP=True."
        )
    package = pip_name or module_name
    print(f"Installing optional Matting Anything dependency: {package}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])


def ensure_matting_anything_assets():
    if not ENABLE_MATTING_ANYTHING:
        return False
    if not MAM_REPO_DIR.exists():
        if not MAM_AUTO_SETUP:
            raise FileNotFoundError(f"Matting Anything repo not found: {MAM_REPO_DIR}")
        MAM_REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        print("Cloning Matting Anything into:", MAM_REPO_DIR)
        subprocess.check_call(
            ["git", "clone", "--depth", "1", "https://github.com/SHI-Labs/Matting-Anything", str(MAM_REPO_DIR)]
        )

    install_python_package_if_missing("easydict", "easydict")
    install_python_package_if_missing("tensorboardX", "tensorboardX")
    install_python_package_if_missing("toml", "toml")

    if not MAM_CHECKPOINT_PATH.exists():
        if not MAM_AUTO_SETUP:
            raise FileNotFoundError(f"Matting Anything checkpoint not found: {MAM_CHECKPOINT_PATH}")
        print("Downloading Matting Anything MAM checkpoint:", MAM_HF_REPO_ID, MAM_HF_FILENAME)
        from huggingface_hub import hf_hub_download

        MAM_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
        downloaded = hf_hub_download(
            repo_id=MAM_HF_REPO_ID,
            filename=MAM_HF_FILENAME,
            local_dir=str(MAM_CHECKPOINT_PATH.parent),
            local_dir_use_symlinks=False,
        )
        downloaded = Path(downloaded)
        if downloaded.resolve() != MAM_CHECKPOINT_PATH.resolve():
            MAM_CHECKPOINT_PATH.write_bytes(downloaded.read_bytes())

    if not MAM_SAM_CHECKPOINT_PATH.exists():
        if not MAM_AUTO_SETUP:
            raise FileNotFoundError(f"SAM ViT-B checkpoint for MAM not found: {MAM_SAM_CHECKPOINT_PATH}")
        MAM_SAM_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
        print("Downloading SAM ViT-B checkpoint for Matting Anything:", MAM_SAM_CHECKPOINT_PATH)
        torch.hub.download_url_to_file(
            MAM_SAM_CHECKPOINT_URL,
            str(MAM_SAM_CHECKPOINT_PATH),
            progress=True,
        )
    return True


def get_matting_anything_model():
    global MAM_MODEL_CACHE
    if MAM_MODEL_CACHE is not None:
        return MAM_MODEL_CACHE
    if not ensure_matting_anything_assets():
        return None

    mam_segment_dir = MAM_REPO_DIR / "segment-anything"
    for path_item in [str(mam_segment_dir), str(MAM_REPO_DIR)]:
        if path_item not in sys.path:
            sys.path.insert(0, path_item)

    for module_name in ["networks", "utils"]:
        module = sys.modules.get(module_name)
        module_file = Path(getattr(module, "__file__", "")) if module is not None else None
        if module_file is not None and MAM_REPO_DIR not in module_file.parents:
            sys.modules.pop(module_name, None)

    import importlib
    import toml
    from segment_anything.utils.transforms import ResizeLongestSide

    cwd = Path.cwd()
    try:
        # MAM's generator uses relative paths for the SAM checkpoint, so build it from the repo root.
        os.chdir(MAM_REPO_DIR)
        mam_utils = importlib.import_module("utils")
        mam_networks = importlib.import_module("networks")
        mam_utils.load_config(toml.load(MAM_CONFIG_PATH))
        model = mam_networks.get_generator_m2m(seg="sam_vit_b", m2m="sam_decoder_deep")
        checkpoint = torch.load(MAM_CHECKPOINT_PATH, map_location="cuda")
        model.m2m.load_state_dict(mam_utils.remove_prefix_state_dict(checkpoint["state_dict"]), strict=True)
        model = model.cuda().eval()
    finally:
        os.chdir(cwd)

    transform = ResizeLongestSide(1024)
    MAM_MODEL_CACHE = {
        "model": model,
        "utils": mam_utils,
        "transform": transform,
    }
    return MAM_MODEL_CACHE


def build_mam_box_sample(image, bbox, transform):
    original_size = image.shape[:2]
    resized = transform.apply_image(image)
    tensor = torch.as_tensor(resized, device="cuda")
    tensor = tensor.permute(2, 0, 1).contiguous().float()
    pixel_mean = torch.tensor([123.675, 116.28, 103.53], device="cuda").view(3, 1, 1)
    pixel_std = torch.tensor([58.395, 57.12, 57.375], device="cuda").view(3, 1, 1)
    tensor = (tensor - pixel_mean) / pixel_std
    h, w = tensor.shape[-2:]
    tensor = torch.nn.functional.pad(tensor, (0, 1024 - w, 0, 1024 - h))

    bbox_arr = np.array([bbox], dtype=np.float32)
    transformed_bbox = transform.apply_boxes(bbox_arr, original_size)[0]
    bbox_tensor = torch.as_tensor(transformed_bbox, dtype=torch.float32, device="cuda")
    return {
        "image": tensor.unsqueeze(0),
        "bbox": bbox_tensor.unsqueeze(0),
        "ori_shape": original_size,
        "pad_shape": (h, w),
    }


def run_matting_anything_box_alpha(image, bbox):
    bundle = get_matting_anything_model()
    if bundle is None:
        return None
    model = bundle["model"]
    mam_utils = bundle["utils"]
    transform = bundle["transform"]
    sample = build_mam_box_sample(image, bbox, transform)

    with torch.no_grad():
        _, pred, post_mask = model.forward_inference(sample)
        alpha_os1 = pred["alpha_os1"][..., : sample["pad_shape"][0], : sample["pad_shape"][1]]
        alpha_os4 = pred["alpha_os4"][..., : sample["pad_shape"][0], : sample["pad_shape"][1]]
        alpha_os8 = pred["alpha_os8"][..., : sample["pad_shape"][0], : sample["pad_shape"][1]]
        alpha_os8 = torch.nn.functional.interpolate(alpha_os8, sample["ori_shape"], mode="bilinear", align_corners=False)
        alpha_os4 = torch.nn.functional.interpolate(alpha_os4, sample["ori_shape"], mode="bilinear", align_corners=False)
        alpha_os1 = torch.nn.functional.interpolate(alpha_os1, sample["ori_shape"], mode="bilinear", align_corners=False)

        # Same refinement schedule as the official demo's alpha guidance path.
        weight_os8 = mam_utils.get_unknown_box_from_mask(post_mask)
        alpha_os8[weight_os8 > 0] = post_mask[weight_os8 > 0]
        alpha_pred = alpha_os8.clone().detach()
        weight_os4 = mam_utils.get_unknown_tensor_from_pred_oneside(alpha_pred, rand_width=20, train_mode=False)
        alpha_pred[weight_os4 > 0] = alpha_os4[weight_os4 > 0]
        weight_os1 = mam_utils.get_unknown_tensor_from_pred_oneside(alpha_pred, rand_width=10, train_mode=False)
        alpha_pred[weight_os1 > 0] = alpha_os1[weight_os1 > 0]

    alpha = alpha_pred[0, 0].detach().cpu().numpy().astype(np.float32)
    return np.clip(alpha, 0.0, 1.0)


def matting_alpha_to_edge_export(conservative_mask, mam_alpha):
    conservative_mask = conservative_mask.astype(bool)
    core_kernel = ensure_odd_kernel(MAM_CORE_ERODE_KERNEL, minimum=1)
    core_mask = cv2.erode(conservative_mask.astype(np.uint8), np.ones((core_kernel, core_kernel), np.uint8), iterations=1).astype(bool)
    if core_mask.sum() < max(25, int(round(conservative_mask.sum() * MAM_MIN_KEEP_RATIO))):
        fallback_kernel = ensure_odd_kernel(max(3, core_kernel // 2), minimum=3)
        core_mask = cv2.erode(conservative_mask.astype(np.uint8), np.ones((fallback_kernel, fallback_kernel), np.uint8), iterations=1).astype(bool)
    if core_mask.sum() < 25:
        core_mask = conservative_mask.copy()

    edge_band = np.logical_and(conservative_mask, ~core_mask)
    alpha = np.zeros(conservative_mask.shape, dtype=np.float32)
    alpha[core_mask] = 1.0

    mam_alpha = np.clip(mam_alpha.astype(np.float32), 0.0, 1.0)
    if MAM_ALPHA_GAMMA != 1.0:
        mam_alpha = np.power(mam_alpha, float(MAM_ALPHA_GAMMA))
    alpha[edge_band] = mam_alpha[edge_band]
    alpha[alpha < float(MAM_SUPPORT_ALPHA_THRESHOLD)] = 0.0
    alpha[core_mask] = 1.0

    support_mask = np.logical_or(core_mask, alpha >= float(MAM_SUPPORT_ALPHA_THRESHOLD))
    support_mask = np.logical_and(support_mask, conservative_mask)
    support_mask = refine_binary_mask(support_mask, close_kernel=3, open_kernel=0)
    if support_mask.sum() < max(25, int(round(conservative_mask.sum() * MAM_MIN_KEEP_RATIO))):
        return None

    boundary_ring = np.logical_and(support_mask, ~core_mask)
    alpha[~support_mask] = 0.0
    return {
        "raw_alpha": mam_alpha,
        "alpha": alpha,
        "support_mask": support_mask,
        "core_mask": core_mask,
        "edge_band": edge_band,
        "boundary_ring": boundary_ring,
    }


def refine_replacement_with_matting_anything(image, bbox, conservative_mask):
    if not ENABLE_MATTING_ANYTHING:
        return None
    try:
        raw_alpha = run_matting_anything_box_alpha(image, bbox)
        pack = matting_alpha_to_edge_export(conservative_mask, raw_alpha)
        if pack is None:
            print("Matting Anything skipped: edge alpha became too small after conservative fusion.")
            return None
        return pack
    except Exception as exc:
        print("Matting Anything skipped:", type(exc).__name__, exc)
        return None


DOG_KEYPOINT_NAMES = [
    "front_left_paw",
    "front_left_knee",
    "front_left_elbow",
    "rear_left_paw",
    "rear_left_knee",
    "rear_left_elbow",
    "front_right_paw",
    "front_right_knee",
    "front_right_elbow",
    "rear_right_paw",
    "rear_right_knee",
    "rear_right_elbow",
    "tail_start",
    "tail_end",
    "left_ear_base",
    "right_ear_base",
    "nose",
    "chin",
    "left_ear_tip",
    "right_ear_tip",
    "left_eye",
    "right_eye",
    "withers",
    "throat",
]
NAME_TO_INDEX = {name: idx for idx, name in enumerate(DOG_KEYPOINT_NAMES)}
KEYPOINT_CONF = 0.20
POSE_CONF = 0.15
POSE_IMGSZ = 960
FULL_PART_SIGNATURE = "head+torso+front_legs+hind_legs+tail"


PART_GROUPS = {
    "head": ["nose", "chin", "left_eye", "right_eye", "left_ear_base", "right_ear_base", "left_ear_tip", "right_ear_tip"],
    "torso": ["withers", "throat", "tail_start"],
    "front_legs": ["front_left_paw", "front_left_knee", "front_left_elbow", "front_right_paw", "front_right_knee", "front_right_elbow"],
    "hind_legs": ["rear_left_paw", "rear_left_knee", "rear_left_elbow", "rear_right_paw", "rear_right_knee", "rear_right_elbow"],
    "tail": ["tail_start", "tail_end"],
}


def count_visible(points, confs, names, threshold=KEYPOINT_CONF):
    return int(sum(float(confs[NAME_TO_INDEX[name]]) >= threshold for name in names))


def visible_body_parts(points, confs):
    head_count = count_visible(points, confs, PART_GROUPS["head"])
    torso_count = count_visible(points, confs, PART_GROUPS["torso"])
    front_count = count_visible(points, confs, PART_GROUPS["front_legs"])
    hind_count = count_visible(points, confs, PART_GROUPS["hind_legs"])
    tail_count = count_visible(points, confs, PART_GROUPS["tail"])

    parts = []
    if head_count >= 2:
        parts.append("head")
    inferred_torso = torso_count >= 1 or (
        (front_count >= 2 and hind_count >= 2)
        or (head_count >= 2 and tail_count >= 1 and (front_count >= 2 or hind_count >= 2))
    )
    if inferred_torso:
        parts.append("torso")
    if front_count >= 2:
        parts.append("front_legs")
    if hind_count >= 2:
        parts.append("hind_legs")
    if tail_count >= 1:
        parts.append("tail")
    return parts


def visible_parts_signature(parts):
    return "+".join(parts) if parts else "none"


def coverage_class_from_parts(parts):
    part_set = set(parts)
    has_head = "head" in part_set
    has_torso = "torso" in part_set
    has_front = "front_legs" in part_set
    has_hind = "hind_legs" in part_set
    has_tail = "tail" in part_set

    if has_head and has_front and has_hind:
        return "full_body"
    if has_head and has_front and not has_hind:
        return "front_half"
    if has_hind and (has_tail or has_torso) and not has_head:
        return "rear_half"
    if has_head and not (has_torso or has_front or has_hind):
        return "head_only"
    if has_head and has_torso and not (has_front or has_hind):
        return "upper_body"
    if has_front and has_hind and not has_head:
        return "body_without_head"
    if has_torso and not (has_head or has_front or has_hind):
        return "torso_only"
    return "misc_partial"


def pose_geometry_label(points, confs):
    nose = named_point(points, confs, "nose")
    tail_start = named_point(points, confs, "tail_start")
    if nose is None or tail_start is None:
        return "unknown"
    dx = float(abs(nose[0] - tail_start[0]))
    dy = float(abs(nose[1] - tail_start[1]))
    if dx < 1e-6:
        return "vertical"
    slope = dy / dx
    if slope < 0.22:
        return "horizontal"
    if slope < 0.55:
        return "diagonal"
    return "vertical"


def get_keypoint_arrays(result, instance_idx):
    xy = result.keypoints.xy[instance_idx].detach().cpu().numpy().astype(np.float32)
    conf_tensor = getattr(result.keypoints, "conf", None)
    if conf_tensor is not None:
        conf = conf_tensor[instance_idx].detach().cpu().numpy().astype(np.float32)
    else:
        data = result.keypoints.data[instance_idx].detach().cpu().numpy().astype(np.float32)
        conf = data[:, 2] if data.shape[-1] >= 3 else np.ones(len(xy), dtype=np.float32)
    return xy, conf


def named_point(points, confs, name, threshold=KEYPOINT_CONF):
    idx = NAME_TO_INDEX[name]
    if confs[idx] >= threshold:
        return points[idx]
    return None


def orientation_label(points, confs):
    nose = named_point(points, confs, "nose")
    tail_start = named_point(points, confs, "tail_start")
    if nose is not None and tail_start is not None:
        return "right" if nose[0] > tail_start[0] else "left"
    left_eye = named_point(points, confs, "left_eye")
    right_eye = named_point(points, confs, "right_eye")
    if left_eye is not None and right_eye is not None:
        return "frontal_or_rear"
    return "unknown"


def classify_view(points, confs):
    orientation = orientation_label(points, confs)
    left_eye = named_point(points, confs, "left_eye")
    right_eye = named_point(points, confs, "right_eye")
    if orientation in {"left", "right"}:
        return "side_profile"
    if left_eye is not None and right_eye is not None:
        return "frontal_or_rear"
    return "ambiguous"


def choose_best_pose_instance(result):
    if result.boxes is None or result.keypoints is None or len(result.boxes) == 0:
        return None
    candidates = []
    for idx, box in enumerate(result.boxes):
        xyxy = box.xyxy[0].detach().cpu().numpy().astype(np.float32)
        area = max(1.0, (xyxy[2] - xyxy[0]) * (xyxy[3] - xyxy[1]))
        score = float(box.conf.item())
        visible = float((get_keypoint_arrays(result, idx)[1] >= KEYPOINT_CONF).sum())
        candidates.append((idx, score * math.sqrt(area) * (1.0 + visible / 24.0)))
    return max(candidates, key=lambda item: item[1])[0]


def detect_pose_record(image):
    results = pose_model.predict(image, conf=POSE_CONF, imgsz=POSE_IMGSZ, device=DEVICE, verbose=False)
    result = results[0]
    best_idx = choose_best_pose_instance(result)
    if best_idx is None:
        raise RuntimeError("Pose model did not return a usable dog instance.")
    bbox = clip_bbox(result.boxes[best_idx].xyxy[0].detach().cpu().numpy().tolist(), image.shape[1], image.shape[0])
    keypoints_xy, keypoints_conf = get_keypoint_arrays(result, best_idx)
    parts = visible_body_parts(keypoints_xy, keypoints_conf)
    return {
        "bbox_xyxy": bbox,
        "detection_conf": round(float(result.boxes[best_idx].conf.item()), 6),
        "points": keypoints_xy,
        "confs": keypoints_conf,
        "visible_keypoints": int((keypoints_conf >= KEYPOINT_CONF).sum()),
        "orientation": orientation_label(keypoints_xy, keypoints_conf),
        "view_class": classify_view(keypoints_xy, keypoints_conf),
        "visible_body_parts": parts,
        "visible_parts_signature": visible_parts_signature(parts),
        "coverage_class": coverage_class_from_parts(parts),
        "pose_geometry_label": pose_geometry_label(keypoints_xy, keypoints_conf),
    }


def serialize_pose_record(pose_record):
    return {
        "bbox_xyxy": [int(v) for v in pose_record["bbox_xyxy"]],
        "detection_conf": round(float(pose_record["detection_conf"]), 6),
        "visible_keypoints": int(pose_record["visible_keypoints"]),
        "orientation": pose_record["orientation"],
        "view_class": pose_record["view_class"],
        "visible_body_parts": list(pose_record.get("visible_body_parts", [])),
        "visible_parts_signature": pose_record.get("visible_parts_signature", "none"),
        "coverage_class": pose_record.get("coverage_class", "unknown"),
        "pose_geometry_label": pose_record.get("pose_geometry_label", "unknown"),
        "points": [[round(float(x), 3), round(float(y), 3)] for x, y in pose_record["points"].tolist()],
        "confs": [round(float(v), 6) for v in pose_record["confs"].tolist()],
    }


def build_pose_canvas(crop_rgb, crop_mask, pad=40):
    h, w = crop_rgb.shape[:2]
    canvas = np.full((h + 2 * pad, w + 2 * pad, 3), 255, dtype=np.uint8)
    masked_crop = crop_rgb.copy()
    masked_crop[~crop_mask] = 255
    canvas[pad:pad + h, pad:pad + w] = masked_crop
    return canvas, pad, pad


def translate_pose_record(pose_record, dx=0.0, dy=0.0):
    translated = {
        **pose_record,
        "bbox_xyxy": [
            int(round(pose_record["bbox_xyxy"][0] + dx)),
            int(round(pose_record["bbox_xyxy"][1] + dy)),
            int(round(pose_record["bbox_xyxy"][2] + dx)),
            int(round(pose_record["bbox_xyxy"][3] + dy)),
        ],
        "points": pose_record["points"].copy(),
        "confs": pose_record["confs"].copy(),
    }
    translated["points"][:, 0] += float(dx)
    translated["points"][:, 1] += float(dy)
    return translated


def make_pose_prior_record(source_entry, side):
    pose_data = source_entry.get(side, {}).get("pose_record")
    if not pose_data:
        return None
    points = np.array(pose_data["points"], dtype=np.float32)
    confs = np.array(pose_data["confs"], dtype=np.float32)
    parts = pose_data.get("visible_body_parts")
    if parts is None:
        parts = visible_body_parts(points, confs)
    return {
        "bbox_xyxy": [int(v) for v in pose_data["bbox_xyxy"]],
        "detection_conf": float(pose_data["detection_conf"]),
        "visible_keypoints": int(pose_data["visible_keypoints"]),
        "orientation": pose_data["orientation"],
        "view_class": pose_data["view_class"],
        "visible_body_parts": list(parts),
        "visible_parts_signature": pose_data.get("visible_parts_signature", visible_parts_signature(parts)),
        "coverage_class": pose_data.get("coverage_class", coverage_class_from_parts(parts)),
        "pose_geometry_label": pose_data.get("pose_geometry_label", pose_geometry_label(points, confs)),
        "points": points,
        "confs": confs,
    }


def mask_keypoint_coverage(mask, pose_record, slack=6):
    if pose_record is None:
        return 0.0
    h, w = mask.shape[:2]
    covered = 0
    total = 0
    for pt, conf in zip(pose_record["points"], pose_record["confs"]):
        if conf < KEYPOINT_CONF:
            continue
        total += 1
        x = int(round(pt[0]))
        y = int(round(pt[1]))
        x1 = max(0, x - slack)
        y1 = max(0, y - slack)
        x2 = min(w, x + slack + 1)
        y2 = min(h, y + slack + 1)
        if x1 < x2 and y1 < y2 and mask[y1:y2, x1:x2].any():
            covered += 1
    return covered / max(1, total)


def build_consensus_mask(mask_a, mask_b):
    if mask_a is None or mask_b is None:
        return None
    overlap = np.logical_and(mask_a, mask_b)
    if overlap.sum() >= 0.35 * min(mask_a.sum(), mask_b.sum()):
        return refine_binary_mask(np.logical_or(mask_a, mask_b), close_kernel=9, open_kernel=3)
    return None


def edge_touch_fraction(mask, width=3):
    return float(mask[:width].mean() + mask[-width:].mean() + mask[:, :width].mean() + mask[:, -width:].mean())


def draw_line_band(mask, p1, p2, radius):
    p1 = tuple(int(round(v)) for v in p1)
    p2 = tuple(int(round(v)) for v in p2)
    cv2.line(mask, p1, p2, 1, thickness=max(1, int(round(radius * 2))))


def draw_point_band(mask, point, radius):
    center = tuple(int(round(v)) for v in point)
    cv2.circle(mask, center, max(1, int(round(radius))), 1, thickness=-1)


POSE_PRIOR_LIMBS = [
    ("nose", "chin"),
    ("nose", "left_ear_base"),
    ("nose", "right_ear_base"),
    ("left_ear_base", "left_ear_tip"),
    ("right_ear_base", "right_ear_tip"),
    ("withers", "tail_start"),
    ("withers", "front_left_elbow"),
    ("front_left_elbow", "front_left_knee"),
    ("front_left_knee", "front_left_paw"),
    ("withers", "front_right_elbow"),
    ("front_right_elbow", "front_right_knee"),
    ("front_right_knee", "front_right_paw"),
    ("tail_start", "rear_left_elbow"),
    ("rear_left_elbow", "rear_left_knee"),
    ("rear_left_knee", "rear_left_paw"),
    ("tail_start", "rear_right_elbow"),
    ("rear_right_elbow", "rear_right_knee"),
    ("rear_right_knee", "rear_right_paw"),
    ("tail_start", "tail_end"),
]




POSE_VISUAL_LIMBS = [
    ("nose", "chin"),
    ("nose", "left_ear_base"),
    ("nose", "right_ear_base"),
    ("left_ear_base", "left_ear_tip"),
    ("right_ear_base", "right_ear_tip"),
    ("withers", "tail_start"),
    ("tail_start", "tail_end"),
    ("withers", "front_left_elbow"),
    ("front_left_elbow", "front_left_knee"),
    ("front_left_knee", "front_left_paw"),
    ("withers", "front_right_elbow"),
    ("front_right_elbow", "front_right_knee"),
    ("front_right_knee", "front_right_paw"),
    ("tail_start", "rear_left_elbow"),
    ("rear_left_elbow", "rear_left_knee"),
    ("rear_left_knee", "rear_left_paw"),
    ("tail_start", "rear_right_elbow"),
    ("rear_right_elbow", "rear_right_knee"),
    ("rear_right_knee", "rear_right_paw"),
]


def draw_pose_record(image, pose_record, title_lines=None):
    out = image.copy()
    if pose_record is None:
        return out
    points = pose_record["points"]
    confs = pose_record["confs"]
    for name_a, name_b in POSE_VISUAL_LIMBS:
        idx_a = NAME_TO_INDEX[name_a]
        idx_b = NAME_TO_INDEX[name_b]
        if confs[idx_a] >= KEYPOINT_CONF and confs[idx_b] >= KEYPOINT_CONF:
            p1 = tuple(int(round(v)) for v in points[idx_a])
            p2 = tuple(int(round(v)) for v in points[idx_b])
            cv2.line(out, p1, p2, (0, 220, 255), 3, lineType=cv2.LINE_AA)
    for name, point, conf in zip(DOG_KEYPOINT_NAMES, points, confs):
        if conf < KEYPOINT_CONF:
            continue
        center = tuple(int(round(v)) for v in point)
        cv2.circle(out, center, 5, (255, 80, 80), -1, lineType=cv2.LINE_AA)
        cv2.putText(out, name, (center[0] + 6, center[1] - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(out, name, (center[0] + 6, center[1] - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (20, 20, 20), 1, cv2.LINE_AA)
    if title_lines:
        y = 24
        for line in title_lines:
            cv2.putText(out, line, (12, y), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 3, cv2.LINE_AA)
            cv2.putText(out, line, (12, y), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (20, 20, 20), 1, cv2.LINE_AA)
            y += 26
    return out


def pose_summary_lines(pose_record):
    if pose_record is None:
        return ["pose: not detected"]
    return [
        f"view: {pose_record.get('view_class', 'unknown')} | orient: {pose_record.get('orientation', 'unknown')}",
        f"coverage: {pose_record.get('coverage_class', 'unknown')}",
        f"parts: {pose_record.get('visible_parts_signature', 'none')}",
        f"visible keypoints: {pose_record.get('visible_keypoints', 0)}",
    ]


def safe_detect_pose_record(image, label):
    try:
        return detect_pose_record(image)
    except Exception as exc:
        print(f"{label} pose labeling skipped:", type(exc).__name__, exc)
        return None


def stable_visible_points(pose_record, names):
    if pose_record is None:
        return []
    pts = []
    for name in names:
        idx = NAME_TO_INDEX[name]
        conf = float(pose_record["confs"][idx])
        if conf >= KEYPOINT_CONF:
            pts.append(pose_record["points"][idx].astype(np.float32))
    return pts


def stable_face_crop_points(pose_record, image_shape, min_side=96):
    """Return robust face-crop anchors that match notebook 02.

    The DragGAN face branch should not let unstable ear-tip/throat points pull the
    crop away from the muzzle. This preview uses the same stable anchors that 02
    uses: nose, chin, eyes, ear bases, plus synthetic cheek anchors when the face
    has only a few reliable points.
    """
    if pose_record is None:
        return None
    h, w = image_shape[:2]
    stable_names = ["nose", "chin", "left_eye", "right_eye", "left_ear_base", "right_ear_base"]
    point_lookup = {}
    for name in stable_names:
        idx = NAME_TO_INDEX[name]
        if float(pose_record["confs"][idx]) >= KEYPOINT_CONF:
            point_lookup[name] = pose_record["points"][idx].astype(np.float32)

    anchors = [pt.copy() for pt in point_lookup.values()]
    nose = point_lookup.get("nose")
    chin = point_lookup.get("chin")
    if nose is not None:
        anchors.append(nose.copy())
        if chin is not None:
            face_len = max(float(np.linalg.norm(chin - nose)), float(min_side) * 0.45)
        else:
            face_len = float(min_side) * 0.55
        anchors.extend([
            nose + np.asarray([-face_len * 0.70, face_len * 0.15], dtype=np.float32),
            nose + np.asarray([face_len * 0.70, face_len * 0.15], dtype=np.float32),
            nose + np.asarray([0.0, -face_len * 0.55], dtype=np.float32),
        ])
        if chin is not None:
            anchors.append(chin.copy())

    if len(anchors) < 2:
        return None
    pts = np.asarray(anchors, dtype=np.float32)
    pts[:, 0] = np.clip(pts[:, 0], 0, w - 1)
    pts[:, 1] = np.clip(pts[:, 1], 0, h - 1)
    return pts


def square_face_crop_box_from_pose(pose_record, image_shape, scale=1.30, min_side=96):
    pts = stable_face_crop_points(pose_record, image_shape, min_side=min_side)
    if pts is None:
        return None
    h, w = image_shape[:2]
    x1, y1 = pts.min(axis=0)
    x2, y2 = pts.max(axis=0)
    cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
    side = max(float(x2 - x1), float(y2 - y1), float(min_side)) * float(scale)
    raw = [
        int(round(cx - side / 2.0)),
        int(round(cy - side / 2.0)),
        int(round(cx + side / 2.0)),
        int(round(cy + side / 2.0)),
    ]
    clipped = [max(0, raw[0]), max(0, raw[1]), min(w, raw[2]), min(h, raw[3])]
    return raw, clipped


def draw_face_crop_preview(image, pose_record, title_lines=None, scale=1.30):
    out = draw_pose_record(image, pose_record, title_lines=title_lines)
    face_box = square_face_crop_box_from_pose(pose_record, image.shape, scale=scale)
    if face_box is None:
        cv2.putText(out, "face crop: unavailable", (12, image.shape[0] - 18), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 3, cv2.LINE_AA)
        cv2.putText(out, "face crop: unavailable", (12, image.shape[0] - 18), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (20, 20, 20), 1, cv2.LINE_AA)
        return out, None
    raw, clipped = face_box
    x1, y1, x2, y2 = clipped
    cv2.rectangle(out, (x1, y1), (x2, y2), (80, 255, 120), 3, cv2.LINE_AA)
    label = f"02 face crop preview raw={raw}"
    cv2.putText(out, label, (12, image.shape[0] - 18), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 3, cv2.LINE_AA)
    cv2.putText(out, label, (12, image.shape[0] - 18), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (20, 20, 20), 1, cv2.LINE_AA)
    return out, {"raw_xyxy": raw, "clipped_xyxy": clipped, "scale": float(scale)}


def estimate_replacement_bbox_prior(image_shape, bbox, return_info=False):
    h, w = image_shape[:2]
    x1, y1, x2, y2 = [int(v) for v in bbox]
    prior = np.zeros((h, w), dtype=bool)
    prior[y1:y2, x1:x2] = True
    info = {"mode": "bbox_rectangle_prior"}
    return (prior, info) if return_info else prior


## Interactive Upload For Notebook 01

Upload one original dog image and one replacement-dog image. When you activate the uploaded pair, notebook `01` will use it, and notebooks `02` and `03` will automatically follow the latest output from `01`.


In [ ]:
upload_status_output = widgets.Output()

org_upload = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload original image",
)
rep_upload = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload replacement image",
)
activate_upload_button = widgets.Button(
    description="Use uploaded pair for 01",
    button_style="primary",
    icon="upload",
)
reset_to_examples_button = widgets.Button(
    description="Reset to example pairs",
    button_style="warning",
    icon="undo",
)


def on_activate_uploaded_pair(_):
    org_entry = extract_upload_entry(org_upload.value)
    rep_entry = extract_upload_entry(rep_upload.value)
    with upload_status_output:
        clear_output(wait=True)
        if org_entry is None or rep_entry is None:
            print("Please upload both the original image and the replacement image before activating the pair.")
            return
        timestamp = Path.cwd().stem + "_" + str(int(torch.randint(0, 10_000_000, (1,)).item()))
        upload_dir = UPLOADED_INPUT_ROOT / timestamp
        org_path = save_upload_entry(org_entry, upload_dir, "org")
        rep_path = save_upload_entry(rep_entry, upload_dir, "rep")
        manifest = save_active_session_manifest(org_path, rep_path, source_mode="interactive_upload")
        print("Activated uploaded pair for notebook 01.")
        print("Original:", manifest["original_path"])
        print("Replacement:", manifest["replacement_path"])
        print("Notebook 02 and 03 will follow this active session automatically after notebook 01 finishes.")


def on_reset_to_examples(_):
    with upload_status_output:
        clear_output(wait=True)
        if ACTIVE_SESSION_MANIFEST_PATH.exists():
            ACTIVE_SESSION_MANIFEST_PATH.unlink()
            print("Removed the active uploaded-pair manifest. Notebook 01 will use the example pairs again.")
        else:
            print("No active uploaded-pair manifest was present. Notebook 01 already uses the example pairs.")


activate_upload_button.on_click(on_activate_uploaded_pair)
reset_to_examples_button.on_click(on_reset_to_examples)

display(
    widgets.VBox(
        [
            widgets.HTML(
                "<h3 style='margin:0 0 8px 0'>Notebook 01 input mode</h3>"
                "<div style='font-size:14px'>Use an uploaded pair for the current session, or reset to the curated example pairs.</div>"
            ),
            widgets.HBox([org_upload, rep_upload], layout=widgets.Layout(gap="12px")),
            widgets.HBox([activate_upload_button, reset_to_examples_button], layout=widgets.Layout(gap="10px")),
            upload_status_output,
        ],
        layout=widgets.Layout(gap="10px"),
    )
)


## Load The Input Pairs And Inspect Detection Boxes

This step now loops over all discovered example pairs.

For each pair, the notebook:

1. loads the original and replacement images
2. runs the detector on both
3. runs the dog pose model locally on both full images
4. derives body-part labels such as `visible_parts_signature` and `coverage_class`
5. visualizes the selected dog boxes and local pose/body labels

If the bounding boxes or local pose labels are already wrong here, later segmentation and warping quality will not rescue the pair. This local labeling step is intentionally independent of notebook `00`, so uploaded images outside the indexed dataset still get usable pose/body metadata.


In [ ]:
PAIR_SPECS, SKIPPED_PAIR_SPECS = resolve_pair_specs_for_notebook01()
if not PAIR_SPECS:
    raise RuntimeError("No pairs are available for notebook 01. Upload a pair or provide approved example pairs.")

print("Resolved notebook 01 pair specs:")
for spec in PAIR_SPECS:
    print(" -", spec["pair_id"], "|", spec["org_path"].name, "->", spec["rep_path"].name, "| source_mode:", spec.get("source_mode", "unknown"))
if SKIPPED_PAIR_SPECS:
    print("Skipped example pairs because notebook 00.5 rejected them:")
    for spec in SKIPPED_PAIR_SPECS:
        reason = spec.get("approval", {}).get("primary_reason", "not_approved")
        print(" -", spec["pair_id"], "|", spec["org_path"].name, "->", spec["rep_path"].name, "|", reason)

PAIR_CONTEXTS = []

for pair_index, spec in enumerate(PAIR_SPECS, start=1):
    pair_name = build_pair_name(spec["org_path"], spec["rep_path"])
    output_dir = OUTPUT_ROOT / pair_name
    output_dir.mkdir(parents=True, exist_ok=True)

    original_img = read_rgb(spec["org_path"])
    replacement_img = read_rgb(spec["rep_path"])

    orig_bbox, orig_det_conf = detect_best_dog(original_img)
    repl_bbox, repl_det_conf = detect_best_dog(replacement_img)

    source_entry = spec.get("example_manifest_entry", {})
    source_orig_pose = make_pose_prior_record(source_entry, "original")
    source_repl_pose = make_pose_prior_record(source_entry, "replacement")
    orig_image_pose = safe_detect_pose_record(original_img, "Original full-image")
    repl_image_pose = safe_detect_pose_record(replacement_img, "Replacement full-image")

    # Notebook 01 must work for uploaded images outside the indexed dataset.
    # Therefore the local pose labels generated here are the primary labels.
    # Dataset labels from notebook 00 are only kept as a fallback.
    orig_prior_pose = orig_image_pose if orig_image_pose is not None else source_orig_pose
    repl_prior_pose = repl_image_pose if repl_image_pose is not None else source_repl_pose

    print_pair_header(f"[01] Pair {pair_index}/{len(PAIR_SPECS)} :: {pair_name}")
    print("Original path:", spec["org_path"])
    print("Replacement path:", spec["rep_path"])
    print("Source mode:", spec.get("source_mode", "unknown"))
    print("Original image shape:", original_img.shape)
    print("Replacement image shape:", replacement_img.shape)
    print("Original dog bbox:", orig_bbox, "conf=", round(orig_det_conf, 4))
    print("Replacement dog bbox:", repl_bbox, "conf=", round(repl_det_conf, 4))
    if spec.get("approval"):
        print("Approval summary:", spec["approval"].get("primary_reason"), "| recommended transform:", spec["approval"].get("recommended_transform_mode"))
    print("Original local pose labels:", None if orig_prior_pose is None else {
        "view_class": orig_prior_pose.get("view_class"),
        "orientation": orig_prior_pose.get("orientation"),
        "coverage_class": orig_prior_pose.get("coverage_class"),
        "visible_parts_signature": orig_prior_pose.get("visible_parts_signature"),
        "visible_keypoints": orig_prior_pose.get("visible_keypoints"),
    })
    print("Replacement local pose labels:", None if repl_prior_pose is None else {
        "view_class": repl_prior_pose.get("view_class"),
        "orientation": repl_prior_pose.get("orientation"),
        "coverage_class": repl_prior_pose.get("coverage_class"),
        "visible_parts_signature": repl_prior_pose.get("visible_parts_signature"),
        "visible_keypoints": repl_prior_pose.get("visible_keypoints"),
    })

    detection_items = [
        ("Original image", original_img),
        ("Original detection", draw_bbox(original_img, orig_bbox)),
        ("Replacement image", replacement_img),
        ("Replacement detection", draw_bbox(replacement_img, repl_bbox)),
        ("Original local pose/body labels", draw_pose_record(original_img, orig_prior_pose, pose_summary_lines(orig_prior_pose))),
        ("Replacement local pose/body labels", draw_pose_record(replacement_img, repl_prior_pose, pose_summary_lines(repl_prior_pose))),
    ]
    plot_images(detection_items, cols=2, figsize=(16, 12))

    PAIR_CONTEXTS.append(
        {
            "pair_index": pair_index,
            "pair_id": spec["pair_id"],
            "pair_name": pair_name,
            "org_path": spec["org_path"],
            "rep_path": spec["rep_path"],
            "output_dir": output_dir,
            "original_img": original_img,
            "replacement_img": replacement_img,
            "orig_bbox": orig_bbox,
            "repl_bbox": repl_bbox,
            "orig_det_conf": orig_det_conf,
            "repl_det_conf": repl_det_conf,
            "source_entry": source_entry,
            "approval": spec.get("approval"),
            "orig_prior_pose": orig_prior_pose,
            "repl_prior_pose": repl_prior_pose,
            "orig_image_pose": orig_image_pose,
            "repl_image_pose": repl_image_pose,
            "source_orig_pose": source_orig_pose,
            "source_repl_pose": source_repl_pose,
            "source_mode": spec.get("source_mode", "unknown"),
        }
    )


## Segment The Original Image

The original image is usually the harder case because the dog is embedded in a more complex scene.

For that reason, we inspect two segmentation candidates:

- `YOLO-seg`: often tighter around the object
- `SAM`: often more general, but sometimes too willing to absorb background

Both paths are then refined with `GrabCut`, and the notebook exports:

- the final original-dog mask
- the hole mask used for later background repair
- the original scene with the dog removed


In [ ]:
for ctx in PAIR_CONTEXTS:
    original_img = ctx["original_img"]
    orig_bbox = ctx["orig_bbox"]
    orig_prior_pose = ctx.get("orig_prior_pose")

    orig_yolo_mask, _ = segment_with_yolo_seg(original_img, orig_bbox)
    orig_sam_mask, _ = segment_with_sam(original_img, orig_bbox, prior_mask=None)
    orig_consensus_mask = build_consensus_mask(orig_yolo_mask, orig_sam_mask)

    orig_candidates = []
    if orig_yolo_mask is not None:
        orig_candidates.append(("YOLO-seg", orig_yolo_mask))
    if orig_sam_mask is not None:
        orig_candidates.append(("SAM", orig_sam_mask))
    if orig_consensus_mask is not None:
        orig_candidates.append(("Consensus union", orig_consensus_mask))
    if not orig_candidates:
        raise RuntimeError(f"No usable original-dog mask was produced for {ctx['pair_name']}.")

    def original_mask_score(mask):
        local_box = mask_to_bbox(mask)
        if local_box is None:
            return -1e9
        keypoint_cov = mask_keypoint_coverage(mask, orig_prior_pose, slack=7)
        area_ratio = mask.sum() / max(1, bbox_area(orig_bbox))
        area_penalty = abs(area_ratio - 1.0)
        return 2.1 * bbox_iou(local_box, orig_bbox) + 1.6 * keypoint_cov - 0.25 * area_penalty - 0.2 * edge_touch_fraction(mask)

    orig_source, orig_mask_raw = max(orig_candidates, key=lambda item: original_mask_score(item[1]))
    orig_mask = refine_mask_grabcut(original_img, orig_mask_raw, iter_count=5)
    orig_hole_mask = orig_mask.copy()
    orig_background_with_hole = original_img.copy()
    orig_background_with_hole[orig_hole_mask] = [255, 255, 255]

    print_pair_header(f"[01] Original segmentation :: {ctx['pair_name']}")
    print("Chosen original mask source:", orig_source)
    print("Original mask area:", int(orig_mask.sum()))
    print("Original keypoint coverage:", round(mask_keypoint_coverage(orig_mask, orig_prior_pose, slack=7), 4))

    orig_items = [("Original detection", draw_bbox(original_img, orig_bbox))]
    if orig_yolo_mask is not None:
        orig_items.append(("Original YOLO-seg mask", overlay_mask(original_img, orig_yolo_mask)))
    if orig_sam_mask is not None:
        orig_items.append(("Original SAM mask", overlay_mask(original_img, orig_sam_mask, color=(0, 220, 80))))
    if orig_consensus_mask is not None:
        orig_items.append(("Original consensus mask", overlay_mask(original_img, orig_consensus_mask, color=(255, 220, 0))))
    orig_items.extend(
        [
            ("Original refined mask", overlay_mask(original_img, orig_mask, color=(255, 180, 0))),
            ("Original background with hole", orig_background_with_hole),
        ]
    )
    plot_images(orig_items, cols=2, figsize=(16, 16))

    ctx.update(
        {
            "orig_source": orig_source,
            "orig_mask": orig_mask,
            "orig_hole_mask": orig_hole_mask,
            "orig_background_with_hole": orig_background_with_hole,
        }
    )


## Segment The Replacement Image And Export RGBA Foreground

The replacement image often has a more centered dog and a simpler composition. In practice, `YOLO-seg` is treated as the **primary proposal** for the replacement subject whenever it is available and reasonably plausible.

This notebook now uses a **precision-first replacement policy**:

- slightly under-covering the dog is acceptable
- contaminating the cutout with background is not

That design choice matches the downstream pipeline. Small missing boundary details can often be recovered later by trimap refinement and diffusion, but background pixels that enter the cutout usually become much harder to remove after warping and compositing.

The replacement prior is intentionally simple now:

- the detected dog bounding box is used as the foreground prior
- this gives `SAM` the full object region without adding noisy border-color cues
- keypoint and ellipse priors are not used in this stage

The replacement workflow is now:

1. generate a foreground prior directly from the detected dog bounding box
2. generate several mask proposals
3. let `YOLO-seg` act as the default leading candidate
4. use `SAM` and the consensus mask mainly as quality checks and fallbacks
5. score candidates with an explicit background-contamination penalty
6. refine the selected binary mask conservatively
7. convert the refined binary mask into a **trimap**
8. run a second boundary-only refinement pass for diagnostics
9. optionally run **Matting Anything** with the dog bounding box as a prompt
10. trust Matting Anything only in the uncertain edge band, while forcing the inner dog core to stay foreground
11. export both a hard support mask and a soft alpha matte

Matting Anything is used as an edge cleaner, not as the main segmenter. This is important for this project: the dog body should not be re-decided from scratch, but the edge alpha should be allowed to remove grass, pavement, wall color, or other background pixels that leaked into the conservative mask.


In [ ]:

for ctx in PAIR_CONTEXTS:
    replacement_img = ctx["replacement_img"]
    repl_bbox = ctx["repl_bbox"]

    repl_bbox_prior_mask, repl_bbox_prior_info = estimate_replacement_bbox_prior(
        replacement_img.shape,
        repl_bbox,
        return_info=True,
    )
    repl_prior_mask = repl_bbox_prior_mask
    repl_yolo_mask, _ = segment_with_yolo_seg(replacement_img, repl_bbox)
    repl_sam_mask, _ = segment_with_sam(replacement_img, repl_bbox, prior_mask=repl_bbox_prior_mask)
    repl_consensus_mask = build_consensus_mask(repl_yolo_mask, repl_sam_mask)

    repl_candidates = []
    if repl_yolo_mask is not None:
        repl_candidates.append(("YOLO-seg", repl_yolo_mask))
    if repl_sam_mask is not None:
        repl_candidates.append(("SAM + bbox prior", repl_sam_mask))
    if repl_consensus_mask is not None:
        repl_candidates.append(("Consensus union", repl_consensus_mask))
    if not repl_candidates:
        repl_candidates.append(("BBox prior fallback", repl_bbox_prior_mask))

    def replacement_mask_metrics(mask):
        local_box = mask_to_bbox(mask)
        if local_box is None:
            return {
                "score": -1e9,
                "bbox_iou": 0.0,
                "prior_iou": 0.0,
                "prior_precision": 0.0,
                "edge_penalty": 1.0,
                "area_ratio": 0.0,
                "background_penalty": 1.0,
                "oversize_penalty": 1.0,
                "underfill_penalty": 1.0,
            }
        inter = np.logical_and(mask, repl_bbox_prior_mask).sum()
        union = np.logical_or(mask, repl_bbox_prior_mask).sum()
        mask_area = max(1, int(mask.sum()))
        prior_iou = inter / max(1, union)
        prior_precision = inter / mask_area
        edge_penalty = edge_touch_fraction(mask)
        area_ratio = mask_area / max(1, bbox_area(repl_bbox))
        bbox_overlap = bbox_iou(local_box, repl_bbox)
        background_penalty = max(0.0, 1.0 - prior_precision)
        oversize_penalty = max(0.0, area_ratio - 0.96)
        underfill_penalty = max(0.0, 0.48 - area_ratio)
        score = (
            1.95 * bbox_overlap
            + 1.10 * prior_iou
            + 1.35 * prior_precision
            - 1.50 * background_penalty
            - 0.95 * edge_penalty
            - 0.85 * oversize_penalty
            - 0.18 * underfill_penalty
        )
        return {
            "score": score,
            "bbox_iou": bbox_overlap,
            "prior_iou": prior_iou,
            "prior_precision": prior_precision,
            "edge_penalty": edge_touch_fraction(mask),
            "area_ratio": area_ratio,
            "background_penalty": background_penalty,
            "oversize_penalty": oversize_penalty,
            "underfill_penalty": underfill_penalty,
        }

    repl_candidate_metrics = {name: replacement_mask_metrics(mask) for name, mask in repl_candidates}

    if len(repl_candidates) == 1 and repl_candidates[0][0] == "BBox prior fallback":
        repl_fallback_source, repl_fallback_mask = repl_candidates[0]
        repl_fallback_metrics = repl_candidate_metrics[repl_fallback_source]
    else:
        fallback_candidates = [(name, mask) for name, mask in repl_candidates if name != "YOLO-seg"]
        if fallback_candidates:
            repl_fallback_source, repl_fallback_mask = max(
                fallback_candidates,
                key=lambda item: repl_candidate_metrics[item[0]]["score"],
            )
            repl_fallback_metrics = repl_candidate_metrics[repl_fallback_source]
        else:
            repl_fallback_source, repl_fallback_mask = "YOLO-seg", repl_yolo_mask
            repl_fallback_metrics = repl_candidate_metrics[repl_fallback_source]

    if repl_yolo_mask is not None:
        repl_yolo_metrics = repl_candidate_metrics["YOLO-seg"]
        yolo_reliable = (
            repl_yolo_metrics["prior_precision"] >= 0.72
            and repl_yolo_metrics["edge_penalty"] <= 0.24
            and repl_yolo_metrics["area_ratio"] >= 0.36
            and repl_yolo_metrics["area_ratio"] <= 1.08
            and repl_yolo_metrics["background_penalty"] <= 0.28
        )
        yolo_competitive = (
            repl_yolo_metrics["score"] >= repl_fallback_metrics["score"] - 0.08
            and repl_yolo_metrics["prior_precision"] >= repl_fallback_metrics["prior_precision"] - 0.08
        )
        if yolo_reliable or yolo_competitive:
            repl_source, repl_mask_raw = "YOLO-seg", repl_yolo_mask
            repl_selection_reason = (
                "YOLO-seg remained the primary replacement proposal under the bbox-prior, precision-first gate. "
                "The mask stayed clean relative to the bbox prior."
            )
        else:
            repl_source, repl_mask_raw = repl_fallback_source, repl_fallback_mask
            repl_selection_reason = (
                "YOLO-seg was available, but a fallback candidate aligned better with the bbox prior "
                "while keeping lower background contamination."
            )
    else:
        repl_source, repl_mask_raw = repl_fallback_source, repl_fallback_mask
        repl_selection_reason = "YOLO-seg was unavailable, so the best fallback candidate was selected."

    repl_mask_refined = refine_replacement_mask_conservative(replacement_img, repl_mask_raw, iter_count=3)

    # Boundary refinement is intentionally separated from coarse segmentation.
    # The coarse mask only says "the dog is roughly here". The trimap then
    # identifies which pixels are definitely foreground, definitely background,
    # and which narrow band still needs fine edge reasoning.
    repl_trimap_pack = refine_replacement_mask_with_trimap(replacement_img, repl_mask_refined)
    repl_trimap = repl_trimap_pack["trimap"]
    repl_mask_boundary_refined = repl_trimap_pack["refined_mask"]
    repl_trimap_support_mask = repl_trimap_pack["support_mask"]
    repl_trimap_core_mask_full = repl_trimap_pack["core_mask"]
    repl_trimap_boundary_ring_full = repl_trimap_pack["boundary_ring"]
    repl_trimap_alpha_soft_full = repl_trimap_pack["soft_alpha"]
    repl_soft_unknown_band = repl_trimap_pack["soft_unknown_band"]
    repl_trimap_seed_band = repl_trimap_pack["unknown_seed_band"]

    # Final replacement export starts from the conservative-refined mask.
    # Matting Anything is then allowed to adjust only the uncertain edge band.
    # This keeps the body core stable while removing background-colored edge leakage.
    repl_mam_pack = refine_replacement_with_matting_anything(
        replacement_img,
        repl_bbox,
        repl_mask_refined,
    )
    repl_export_mask_source = "conservative_refined"
    repl_alpha_override_full = None
    repl_core_override_full = None
    repl_boundary_override_full = None
    if repl_mam_pack is not None:
        repl_mask = repl_mam_pack["support_mask"]
        repl_alpha_override_full = repl_mam_pack["alpha"]
        repl_core_override_full = repl_mam_pack["core_mask"]
        repl_boundary_override_full = repl_mam_pack["boundary_ring"]
        repl_export_mask_source = "matting_anything_edge_alpha"
    else:
        repl_mask = repl_mask_refined.copy()

    repl_extract = extract_rgba(
        replacement_img,
        repl_mask,
        return_details=True,
        decontaminate_edges=True,
        alpha_override=repl_alpha_override_full,
        core_mask_override=repl_core_override_full,
        boundary_ring_override=repl_boundary_override_full,
    )
    repl_fg_rgb = repl_extract["rgb"]
    repl_fg_mask = repl_extract["mask"]
    repl_fg_rgba = repl_extract["rgba"]
    repl_fg_bbox = repl_extract["bbox"]
    repl_core_mask = repl_extract["core_mask"]
    repl_boundary_ring = repl_extract["boundary_ring"]
    repl_alpha_soft = repl_extract["alpha"]
    repl_edge_background_rgb = repl_extract["edge_background_rgb"]
    repl_alpha = repl_fg_rgba[..., 3]

    print_pair_header(f"[01] Replacement segmentation :: {ctx['pair_name']}")
    print("Chosen replacement mask source:", repl_source)
    print("Replacement selection reason:", repl_selection_reason)
    print("Replacement candidate scores:")
    for candidate_name, candidate_metrics in sorted(repl_candidate_metrics.items(), key=lambda item: item[1]["score"], reverse=True):
        print(
            "  -",
            candidate_name,
            {
                "score": round(float(candidate_metrics["score"]), 4),
                "bbox_iou": round(float(candidate_metrics["bbox_iou"]), 4),
                "prior_iou": round(float(candidate_metrics["prior_iou"]), 4),
                "prior_precision": round(float(candidate_metrics["prior_precision"]), 4),
                "edge_penalty": round(float(candidate_metrics["edge_penalty"]), 4),
                "background_penalty": round(float(candidate_metrics["background_penalty"]), 4),
                "area_ratio": round(float(candidate_metrics["area_ratio"]), 4),
            },
        )
    print("Replacement foreground bbox:", repl_fg_bbox)
    print("Replacement conservative-refined mask area:", int(repl_mask_refined.sum()))
    print("Replacement trimap-refined support area:", int(repl_trimap_support_mask.sum()))
    print("Replacement export mask source:", repl_export_mask_source)
    if repl_mam_pack is not None:
        print("Matting Anything raw alpha area:", int((repl_mam_pack["raw_alpha"] >= MAM_SUPPORT_ALPHA_THRESHOLD).sum()))
        print("Matting Anything fused support area:", int(repl_mam_pack["support_mask"].sum()))
    print("Replacement unknown-band area:", int(repl_soft_unknown_band.sum()))
    print("Replacement prior mode:", repl_bbox_prior_info["mode"])
    print("Edge decontamination background RGB:", None if repl_edge_background_rgb is None else [round(v, 2) for v in repl_edge_background_rgb])


    repl_items = [
        ("Replacement detection", draw_bbox(replacement_img, repl_bbox)),
        ("BBox rectangle prior (used)", overlay_mask(replacement_img, repl_bbox_prior_mask, color=(0, 220, 80))),
    ]
    if repl_yolo_mask is not None:
        repl_items.append(("Replacement YOLO-seg", overlay_mask(replacement_img, repl_yolo_mask)))
    if repl_sam_mask is not None:
        repl_items.append(("Replacement SAM", overlay_mask(replacement_img, repl_sam_mask, color=(255, 180, 0))))
    if repl_consensus_mask is not None:
        repl_items.append(("Replacement consensus mask", overlay_mask(replacement_img, repl_consensus_mask, color=(255, 220, 0))))
    repl_items.extend(
        [
            ("Replacement conservative-refined mask", overlay_mask(replacement_img, repl_mask_refined, color=(255, 80, 80))),
            *([] if repl_mam_pack is None else [
                ("MAM raw alpha", np.uint8(np.clip(repl_mam_pack["raw_alpha"], 0.0, 1.0) * 255)),
                ("MAM edge band used", overlay_mask(replacement_img, repl_mam_pack["edge_band"], color=(0, 255, 255))),
                ("MAM fused alpha export", np.uint8(np.clip(repl_mam_pack["alpha"], 0.0, 1.0) * 255)),
                ("MAM fused support mask", overlay_mask(replacement_img, repl_mam_pack["support_mask"], color=(80, 220, 255))),
            ]),
            ("Replacement trimap", visualize_trimap(repl_trimap)),
            ("Replacement trimap seed band", overlay_mask(replacement_img, repl_trimap_seed_band, color=(255, 210, 0))),
            ("Replacement boundary-refined support", overlay_mask(replacement_img, repl_mask, color=(80, 220, 255))),
            ("Replacement core mask", overlay_mask(replacement_img, repl_trimap_core_mask_full, color=(255, 255, 255))),
            ("Replacement boundary ring", overlay_mask(replacement_img, repl_trimap_boundary_ring_full, color=(255, 0, 255))),
            ("Replacement soft alpha", np.uint8(np.clip(repl_trimap_alpha_soft_full, 0.0, 1.0) * 255)),
            ("Replacement soft-unknown band", overlay_mask(replacement_img, repl_soft_unknown_band, color=(255, 120, 0))),
            ("Replacement alpha export", repl_alpha),
        ]
    )
    plot_images(repl_items, cols=2, figsize=(16, 16))
    show_rgba(repl_fg_rgba, title=f"Replacement dog RGBA on checkerboard :: {ctx['pair_name']}")

    ctx.update(
        {
            "repl_bbox_prior_mask": repl_bbox_prior_mask,
            "repl_prior_mask": repl_prior_mask,
            "repl_source": repl_source,
            "repl_selection_reason": repl_selection_reason,
            "repl_candidate_metrics": repl_candidate_metrics,
            "repl_mask_refined": repl_mask_refined,
            "repl_trimap": repl_trimap,
            "repl_mask_boundary_refined": repl_mask_boundary_refined,
            "repl_trimap_seed_band": repl_trimap_seed_band,
            "repl_soft_unknown_band": repl_soft_unknown_band,
            "repl_mask": repl_mask,
            "repl_export_mask_source": repl_export_mask_source,
            "repl_mam_pack": repl_mam_pack,
            "repl_fg_rgb": repl_fg_rgb,
            "repl_fg_mask": repl_fg_mask,
            "repl_fg_rgba": repl_fg_rgba,
            "repl_fg_bbox": repl_fg_bbox,
            "repl_core_mask": repl_core_mask,
            "repl_boundary_ring": repl_boundary_ring,
            "repl_alpha_soft": repl_alpha_soft,
            "repl_trimap_core_mask": repl_trimap_core_mask_full,
            "repl_trimap_boundary_ring": repl_trimap_boundary_ring_full,
            "repl_trimap_alpha_soft": repl_trimap_alpha_soft_full,
            "repl_trimap_support_mask": repl_trimap_support_mask,
            "repl_edge_background_rgb": repl_edge_background_rgb,
            "repl_alpha": repl_alpha,
        }
    )


## Export The Cutout Assets Used By 02 And 03

Each discovered pair now writes its own standardized package to:

- `data/outputs/01_cutout/<pair_name>/`

The export format stays the same as before. The only difference is that the notebook now repeats the same asset package for every configured example pair.


In [ ]:
exported_pairs = []

for ctx in PAIR_CONTEXTS:
    output_dir = ctx["output_dir"]
    original_img = ctx["original_img"]
    replacement_img = ctx["replacement_img"]
    orig_mask = ctx["orig_mask"]
    orig_hole_mask = ctx["orig_hole_mask"]
    orig_background_with_hole = ctx["orig_background_with_hole"]
    repl_mask_refined = ctx.get("repl_mask_refined", ctx["repl_mask"])
    repl_bbox_prior_mask = ctx["repl_bbox_prior_mask"]
    repl_trimap = ctx["repl_trimap"]
    repl_mask_boundary_refined = ctx["repl_mask_boundary_refined"]
    repl_trimap_support_mask = ctx["repl_trimap_support_mask"]
    repl_trimap_seed_band = ctx["repl_trimap_seed_band"]
    repl_soft_unknown_band = ctx["repl_soft_unknown_band"]
    repl_trimap_boundary_ring = ctx["repl_trimap_boundary_ring"]
    repl_mask = ctx["repl_mask"]
    repl_core_mask = ctx["repl_core_mask"]
    repl_boundary_ring = ctx["repl_boundary_ring"]
    repl_alpha_soft = ctx["repl_alpha_soft"]
    repl_edge_background_rgb = ctx.get("repl_edge_background_rgb")
    repl_export_mask_source = ctx.get("repl_export_mask_source", "conservative_refined")
    repl_mam_pack = ctx.get("repl_mam_pack")
    repl_fg_rgb = ctx["repl_fg_rgb"]
    repl_fg_rgba = ctx["repl_fg_rgba"]
    repl_alpha = ctx["repl_alpha"]

    save_rgb(output_dir / "original_image.png", original_img)
    save_rgb(output_dir / "replacement_image.png", replacement_img)

    save_mask(output_dir / "org_mask.png", orig_mask)
    save_mask(output_dir / "org_hole_mask.png", orig_hole_mask)
    save_rgb(output_dir / "org_background_with_hole.png", orig_background_with_hole)

    save_mask(output_dir / "rep_mask_refined.png", repl_mask_refined)
    Image.fromarray(repl_trimap.astype(np.uint8)).save(output_dir / "rep_trimap.png")
    save_mask(output_dir / "rep_boundary_refined_mask.png", repl_mask_boundary_refined)
    save_mask(output_dir / "rep_trimap_seed_band_mask.png", repl_trimap_seed_band)
    save_mask(output_dir / "rep_soft_unknown_band_mask.png", repl_soft_unknown_band)
    save_mask(output_dir / "rep_mask.png", repl_mask)
    save_mask(output_dir / "rep_core_mask.png", repl_core_mask)
    save_mask(output_dir / "rep_boundary_ring_mask.png", repl_boundary_ring)
    if repl_mam_pack is not None:
        Image.fromarray(np.uint8(np.clip(repl_mam_pack["raw_alpha"], 0.0, 1.0) * 255)).save(output_dir / "rep_mam_raw_alpha.png")
        Image.fromarray(np.uint8(np.clip(repl_mam_pack["alpha"], 0.0, 1.0) * 255)).save(output_dir / "rep_mam_fused_alpha.png")
        save_mask(output_dir / "rep_mam_edge_band_mask.png", repl_mam_pack["edge_band"])
        save_mask(output_dir / "rep_mam_support_mask.png", repl_mam_pack["support_mask"])
    save_rgb(output_dir / "rep_rgb_crop.png", repl_fg_rgb)
    Image.fromarray(np.uint8(np.clip(repl_alpha_soft, 0.0, 1.0) * 255)).save(output_dir / "rep_soft_alpha.png")
    Image.fromarray(repl_alpha.astype(np.uint8)).save(output_dir / "rep_alpha.png")
    Image.fromarray(repl_fg_rgba).save(output_dir / "rep_rgba.png")

    org_dog_rgb, org_dog_mask, org_dog_rgba, org_dog_bbox = extract_rgba(original_img, orig_mask)
    save_rgb(output_dir / "org_rgb_crop.png", org_dog_rgb)
    Image.fromarray(org_dog_rgba).save(output_dir / "org_rgba.png")

    org_pose_canvas, org_pad_x, org_pad_y = build_pose_canvas(org_dog_rgb, org_dog_mask)
    rep_pose_canvas, rep_pad_x, rep_pad_y = build_pose_canvas(repl_fg_rgb, ctx["repl_fg_mask"])
    save_rgb(output_dir / "org_pose_canvas.png", org_pose_canvas)
    save_rgb(output_dir / "rep_pose_canvas.png", rep_pose_canvas)

    org_detected_cutout_pose_canvas = safe_detect_pose_record(org_pose_canvas, "Original cutout pose canvas")
    rep_detected_cutout_pose_canvas = safe_detect_pose_record(rep_pose_canvas, "Replacement cutout pose canvas")
    org_detected_cutout_pose = None if org_detected_cutout_pose_canvas is None else translate_pose_record(org_detected_cutout_pose_canvas, dx=-org_pad_x, dy=-org_pad_y)
    rep_detected_cutout_pose = None if rep_detected_cutout_pose_canvas is None else translate_pose_record(rep_detected_cutout_pose_canvas, dx=-rep_pad_x, dy=-rep_pad_y)

    # The full-image pose is usually more stable than re-running pose on a cutout
    # with a white/transparent background. Use the full-image keypoints as the
    # source of truth, then translate them into the exported cutout coordinate
    # system so notebook 02 crops the same face region without re-detecting it.
    org_cutout_pose_source = "full_image_pose_translated_to_cutout" if ctx.get("orig_prior_pose") is not None else "cutout_pose_canvas_detection"
    rep_cutout_pose_source = "full_image_pose_translated_to_cutout" if ctx.get("repl_prior_pose") is not None else "cutout_pose_canvas_detection"
    org_cutout_pose = (
        translate_pose_record(ctx["orig_prior_pose"], dx=-org_dog_bbox[0], dy=-org_dog_bbox[1])
        if ctx.get("orig_prior_pose") is not None else org_detected_cutout_pose
    )
    rep_cutout_pose = (
        translate_pose_record(ctx["repl_prior_pose"], dx=-ctx["repl_fg_bbox"][0], dy=-ctx["repl_fg_bbox"][1])
        if ctx.get("repl_prior_pose") is not None else rep_detected_cutout_pose
    )
    if org_cutout_pose is None or rep_cutout_pose is None:
        raise RuntimeError("Missing usable cutout pose. Check full-image pose and cutout pose-canvas detection.")
    org_full_face_crop_preview, org_full_face_crop_preview_meta = draw_face_crop_preview(
        original_img,
        ctx.get("orig_prior_pose"),
        title_lines=["Original full-image pose face crop source"],
        scale=FACE_CROP_SCALE,
    )
    rep_full_face_crop_preview, rep_full_face_crop_preview_meta = draw_face_crop_preview(
        replacement_img,
        ctx.get("repl_prior_pose"),
        title_lines=["Replacement full-image pose face crop source"],
        scale=FACE_CROP_SCALE,
    )
    org_face_crop_preview, org_face_crop_preview_meta = draw_face_crop_preview(
        org_dog_rgb,
        org_cutout_pose,
        title_lines=["Original full-image pose synced to cutout + 02 face crop"],
        scale=FACE_CROP_SCALE,
    )
    rep_face_crop_preview, rep_face_crop_preview_meta = draw_face_crop_preview(
        repl_fg_rgb,
        rep_cutout_pose,
        title_lines=["Replacement full-image pose synced to cutout + 02 face crop"],
        scale=FACE_CROP_SCALE,
    )
    save_rgb(output_dir / "org_full_image_face_crop_source.png", org_full_face_crop_preview)
    save_rgb(output_dir / "rep_full_image_face_crop_source.png", rep_full_face_crop_preview)
    save_rgb(output_dir / "org_face_crop_preview.png", org_face_crop_preview)
    save_rgb(output_dir / "rep_face_crop_preview.png", rep_face_crop_preview)

    print_pair_header(f"[01] Cutout pose re-check :: {ctx['pair_name']}")
    print("Original cutout pose source:", org_cutout_pose_source)
    print("Replacement cutout pose source:", rep_cutout_pose_source)
    print("Original synced cutout pose:", {k: org_cutout_pose[k] for k in ['view_class', 'orientation', 'visible_keypoints', 'bbox_xyxy']})
    print("Replacement synced cutout pose:", {k: rep_cutout_pose[k] for k in ['view_class', 'orientation', 'visible_keypoints', 'bbox_xyxy']})
    print("Original detected cutout pose:", None if org_detected_cutout_pose is None else {k: org_detected_cutout_pose[k] for k in ['view_class', 'orientation', 'visible_keypoints', 'bbox_xyxy']})
    print("Replacement detected cutout pose:", None if rep_detected_cutout_pose is None else {k: rep_detected_cutout_pose[k] for k in ['view_class', 'orientation', 'visible_keypoints', 'bbox_xyxy']})
    print("Original full-image face crop source:", org_full_face_crop_preview_meta)
    print("Replacement full-image face crop source:", rep_full_face_crop_preview_meta)
    print("Original 02 face crop preview:", org_face_crop_preview_meta)
    print("Replacement 02 face crop preview:", rep_face_crop_preview_meta)

    pose_items = [
        ("Original pose canvas debug", org_pose_canvas),
        ("Replacement pose canvas debug", rep_pose_canvas),
        ("Original full-image face crop source", org_full_face_crop_preview),
        ("Replacement full-image face crop source", rep_full_face_crop_preview),
        ("Original synced full-image pose + 02 face crop", org_face_crop_preview),
        ("Replacement synced full-image pose + 02 face crop", rep_face_crop_preview),
        ("Original refined mask", overlay_mask(original_img, orig_mask, color=(255, 180, 0))),
        ("Replacement bbox prior", overlay_mask(replacement_img, repl_bbox_prior_mask, color=(0, 220, 80))),
                ("Replacement conservative-refined mask", overlay_mask(replacement_img, repl_mask_refined, color=(255, 80, 80))),
        ("Replacement trimap", visualize_trimap(repl_trimap)),
        ("Replacement boundary-refined support", overlay_mask(replacement_img, repl_mask, color=(80, 220, 255))),
        ("Replacement trimap boundary ring", overlay_mask(replacement_img, repl_trimap_boundary_ring, color=(255, 0, 255))),
    ]
    plot_images(pose_items, cols=2, figsize=(16, 12))

    metadata = {
        "pair_id": ctx["pair_id"],
        "pair_name": ctx["pair_name"],
        "pair_index": ctx["pair_index"],
        "project_root": rel_path(PROJECT_ROOT),
        "output_dir": rel_path(output_dir),
        "inputs": {
            "original_path": str(ctx["org_path"]),
            "replacement_path": str(ctx["rep_path"]),
        },
        "source_records": {
            "original_relative_path": ctx.get("source_entry", {}).get("original", {}).get("relative_path"),
            "replacement_relative_path": ctx.get("source_entry", {}).get("replacement", {}).get("relative_path"),
        },
        "approval": ctx.get("approval"),
        "source_mode": ctx.get("source_mode", "unknown"),
        "models": {
            "yolo_det": str(YOLO_DET_WEIGHTS),
            "yolo_seg": str(YOLO_SEG_WEIGHTS),
            "sam": str(SAM_WEIGHTS),
            "pose": str(POSE_MODEL_PATH),
            "matting_anything_enabled": bool(ENABLE_MATTING_ANYTHING),
            "matting_anything_repo": str(MAM_REPO_DIR),
            "matting_anything_checkpoint": str(MAM_CHECKPOINT_PATH),
        },
        "original": {
            "bbox_xyxy": [int(v) for v in ctx["orig_bbox"]],
            "detection_conf": round(float(ctx["orig_det_conf"]), 6),
            "mask_source": ctx["orig_source"],
            "mask_path": rel_path(output_dir / "org_mask.png"),
            "hole_mask_path": rel_path(output_dir / "org_hole_mask.png"),
            "background_with_hole_path": rel_path(output_dir / "org_background_with_hole.png"),
            "rgba_path": rel_path(output_dir / "org_rgba.png"),
            "rgb_crop_path": rel_path(output_dir / "org_rgb_crop.png"),
            "crop_bbox_xyxy": [int(v) for v in org_dog_bbox],
            "pose_canvas_path": rel_path(output_dir / "org_pose_canvas.png"),
            "full_image_face_crop_source_path": rel_path(output_dir / "org_full_image_face_crop_source.png"),
            "full_image_face_crop_source": org_full_face_crop_preview_meta,
            "face_crop_preview_path": rel_path(output_dir / "org_face_crop_preview.png"),
            "face_crop_preview": org_face_crop_preview_meta,
            "cutout_pose_source": org_cutout_pose_source,
            "prior_pose": None if ctx.get("orig_prior_pose") is None else serialize_pose_record(ctx["orig_prior_pose"]),
            "full_image_pose_source": "local_01_pose_model" if ctx.get("orig_image_pose") is not None else "dataset_manifest_fallback",
            "full_image_pose": None if ctx.get("orig_image_pose") is None else serialize_pose_record(ctx["orig_image_pose"]),
            "cutout_pose": serialize_pose_record(org_cutout_pose),
            "detected_cutout_pose": None if org_detected_cutout_pose is None else serialize_pose_record(org_detected_cutout_pose),
        },
        "replacement": {
            "bbox_xyxy": [int(v) for v in ctx["repl_bbox"]],
            "detection_conf": round(float(ctx["repl_det_conf"]), 6),
            "mask_source": ctx["repl_source"],
            "selection_reason": ctx.get("repl_selection_reason"),
            "candidate_metrics": ctx.get("repl_candidate_metrics"),
            "pose_prior_type": "bbox_plus_skeleton",
            "refined_mask_path": rel_path(output_dir / "rep_mask_refined.png"),
            "trimap_path": rel_path(output_dir / "rep_trimap.png"),
            "boundary_refined_mask_path": rel_path(output_dir / "rep_boundary_refined_mask.png"),
            "trimap_seed_band_path": rel_path(output_dir / "rep_trimap_seed_band_mask.png"),
            "soft_unknown_band_path": rel_path(output_dir / "rep_soft_unknown_band_mask.png"),
            "mask_path": rel_path(output_dir / "rep_mask.png"),
            "core_mask_path": rel_path(output_dir / "rep_core_mask.png"),
            "boundary_ring_mask_path": rel_path(output_dir / "rep_boundary_ring_mask.png"),
            "matting_anything_outputs": None if repl_mam_pack is None else {
                "raw_alpha_path": rel_path(output_dir / "rep_mam_raw_alpha.png"),
                "fused_alpha_path": rel_path(output_dir / "rep_mam_fused_alpha.png"),
                "edge_band_path": rel_path(output_dir / "rep_mam_edge_band_mask.png"),
                "support_mask_path": rel_path(output_dir / "rep_mam_support_mask.png"),
            },
            "mask_processing": {
                "conservative_refined_area": int(repl_mask_refined.sum()),
                "boundary_refined_area": int(repl_mask_boundary_refined.sum()),
                "trimap_support_area": int(repl_trimap_support_mask.sum()),
                "support_area": int(repl_mask.sum()),
                "export_mask_source": repl_export_mask_source,
                "matting_anything_used": repl_mam_pack is not None,
                "matting_anything_core_erode_kernel": MAM_CORE_ERODE_KERNEL,
                "matting_anything_support_alpha_threshold": MAM_SUPPORT_ALPHA_THRESHOLD,
                "matting_anything_alpha_gamma": MAM_ALPHA_GAMMA,
                "core_area": int(repl_core_mask.sum()),
                "boundary_ring_area": int(repl_boundary_ring.sum()),
                "soft_unknown_band_area": int(repl_soft_unknown_band.sum()),
                "trimap_fg_kernel": REP_TRIMAP_FG_KERNEL,
                "trimap_bg_kernel": REP_TRIMAP_BG_KERNEL,
                "trimap_iterations": REP_TRIMAP_ITER_COUNT,
                "alpha_threshold": REP_TRIMAP_ALPHA_THRESHOLD,
                "core_kernel": REP_CORE_MASK_KERNEL,
                "edge_decontam_strength": REP_EDGE_DECONTAM_STRENGTH,
                "edge_background_rgb": repl_edge_background_rgb,
            },
            "rgba_path": rel_path(output_dir / "rep_rgba.png"),
            "soft_alpha_path": rel_path(output_dir / "rep_soft_alpha.png"),
            "alpha_path": rel_path(output_dir / "rep_alpha.png"),
            "rgb_crop_path": rel_path(output_dir / "rep_rgb_crop.png"),
            "crop_bbox_xyxy": [int(v) for v in ctx["repl_fg_bbox"]],
            "pose_canvas_path": rel_path(output_dir / "rep_pose_canvas.png"),
            "full_image_face_crop_source_path": rel_path(output_dir / "rep_full_image_face_crop_source.png"),
            "full_image_face_crop_source": rep_full_face_crop_preview_meta,
            "face_crop_preview_path": rel_path(output_dir / "rep_face_crop_preview.png"),
            "face_crop_preview": rep_face_crop_preview_meta,
            "cutout_pose_source": rep_cutout_pose_source,
            "prior_pose": None if ctx.get("repl_prior_pose") is None else serialize_pose_record(ctx["repl_prior_pose"]),
            "full_image_pose_source": "local_01_pose_model" if ctx.get("repl_image_pose") is not None else "dataset_manifest_fallback",
            "full_image_pose": None if ctx.get("repl_image_pose") is None else serialize_pose_record(ctx["repl_image_pose"]),
            "cutout_pose": serialize_pose_record(rep_cutout_pose),
            "detected_cutout_pose": None if rep_detected_cutout_pose is None else serialize_pose_record(rep_detected_cutout_pose),
        },
    }

    with open(output_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print_pair_header(f"[01] Export summary :: {ctx['pair_name']}")
    print("Saved cutout package to:", output_dir)
    print("Metadata file:", output_dir / "metadata.json")

    plot_images(
        [
            ("Original refined mask", overlay_mask(original_img, orig_mask, color=(255, 180, 0))),
            ("Original background with hole", orig_background_with_hole),
            ("Replacement conservative-refined mask", overlay_mask(replacement_img, repl_mask_refined, color=(255, 80, 80))),
            *([] if repl_mam_pack is None else [
                ("MAM fused alpha export", np.uint8(np.clip(repl_mam_pack["alpha"], 0.0, 1.0) * 255)),
                ("MAM fused support mask", overlay_mask(replacement_img, repl_mam_pack["support_mask"], color=(80, 220, 255))),
            ]),
            ("Replacement trimap", visualize_trimap(repl_trimap)),
            ("Replacement boundary-refined support", overlay_mask(replacement_img, repl_mask, color=(80, 220, 255))),
            ("Replacement soft alpha", np.uint8(np.clip(repl_alpha_soft, 0.0, 1.0) * 255)),
            ("Replacement RGBA", rgba_on_checkerboard(repl_fg_rgba)),
        ],
        cols=2,
        figsize=(16, 12),
    )

    exported_pairs.append(
        {
            "pair_id": ctx["pair_id"],
            "pair_name": ctx["pair_name"],
            "pair_index": ctx["pair_index"],
            "output_dir": rel_path(output_dir),
            "metadata_path": rel_path(output_dir / "metadata.json"),
        }
    )

with open(OUTPUT_ROOT / "batch_manifest.json", "w", encoding="utf-8") as f:
    json.dump(exported_pairs, f, ensure_ascii=False, indent=2)
with open(ACTIVE_BATCH_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(exported_pairs, f, ensure_ascii=False, indent=2)

EXPORTED_PAIR_CONTEXTS = PAIR_CONTEXTS
print("\nSaved batch manifest:", OUTPUT_ROOT / "batch_manifest.json")
print("Saved active batch manifest:", ACTIVE_BATCH_MANIFEST_PATH)


In [ ]:
print("Processed pairs:")
for ctx in EXPORTED_PAIR_CONTEXTS:
    print(" -", ctx["pair_index"], ctx["pair_name"], "->", ctx["output_dir"])
